# UTD-MHAD: Comprehensive Per-Fold Feature Selection Comparison

This notebook compares **17 feature selection methods** across all 8 LOSO folds (treated individually):

**Standard Methods:**
1. Mutual Information (filter)
2. RFE (wrapper)
3. LASSO/L1 (embedded)

**14 EvoloPy Metaheuristics:**
BAT, CS, DE, FFA, GA, GWO, HHO, JAYA, MFO, MVO, PSO, SCA, SSA, WOA

All results are saved **per fold** (no averaging) inside `results_per_fold/`.
Comprehensive plots and CSVs are saved inside `results_per_fold/plots/`.


In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
import torch
print(torch.cuda.device_count())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

1
NVIDIA GeForce RTX 4090


In [3]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import time
import warnings
from pathlib import Path
from tqdm import tqdm
from copy import deepcopy
import random
import sys
from scipy import stats
import psutil
import tracemalloc
import gc
import json as json_lib

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score,
                             precision_score, recall_score, classification_report)

# Standard feature selection imports
from sklearn.feature_selection import mutual_info_classif, RFE, SelectKBest
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Import ALL EvoloPy optimizers
sys.path.append('../EvoloPy-master')
from EvoloPy.optimizers import BAT, CS, DE, FFA, GA, GWO, HHO, JAYA, MFO, MVO, PSO, SCA, SSA, WOA

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)
random.seed(42)

# Global device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"PyTorch version: {torch.__version__}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")


Device: cuda
PyTorch version: 2.9.1+cu128
GPU: NVIDIA GeForce RTX 4090


## Configuration

In [4]:
# Paths
FEATURE_DIR = Path("features_new")

# Master output directory
RESULTS_ROOT = Path("results_per_fold_new")
RESULTS_ROOT.mkdir(exist_ok=True)
PLOTS_DIR = RESULTS_ROOT / "plots"
PLOTS_DIR.mkdir(exist_ok=True)

# Hyperparameters
N_FOLDS = 8  # LOSO
N_EPOCHS = 300
BATCH_SIZE = 32
LEARNING_RATE = 0.001
HIDDEN_DIM = 128

# Feature selection hyperparameters - metaheuristic
N_POPULATION = 20
MAX_ITERATIONS = 30

# For standard methods (target: select ~50% of features)
TARGET_FEATURE_PERCENTAGE = 0.5

# All EvoloPy optimizer names and their callable entry points
EVOLOPY_OPTIMIZERS = {
    'BAT': BAT.BAT,
    'CS':  CS.CS,
    'DE':  DE.DE,
    'FFA': FFA.FFA,
    'GA':  GA.GA,
    'GWO': GWO.GWO,
    'HHO': HHO.HHO,
    'JAYA': JAYA.JAYA,
    'MFO': MFO.MFO,
    'MVO': MVO.MVO,
    'PSO': PSO.PSO,
    'SCA': SCA.SCA,
    'SSA': SSA.SSA,
    'WOA': WOA.WOA,
}

ALL_METHODS = ['baseline', 'mutual_info', 'rfe', 'lasso'] + [f'meta_{k}' for k in EVOLOPY_OPTIMIZERS.keys()]

# Create per-method subdirectories
for method in ALL_METHODS:
    (RESULTS_ROOT / method).mkdir(exist_ok=True)

print(f"Configuration:")
print(f"  N_FOLDS: {N_FOLDS}")
print(f"  N_EPOCHS: {N_EPOCHS}")
print(f"  Target feature selection: {TARGET_FEATURE_PERCENTAGE*100:.0f}%")
print(f"  Metaheuristic pop: {N_POPULATION}, iter: {MAX_ITERATIONS}")
print(f"  EvoloPy optimizers: {list(EVOLOPY_OPTIMIZERS.keys())}")
print(f"  Total methods (incl. baseline): {len(ALL_METHODS)}")


Configuration:
  N_FOLDS: 8
  N_EPOCHS: 300
  Target feature selection: 50%
  Metaheuristic pop: 20, iter: 30
  EvoloPy optimizers: ['BAT', 'CS', 'DE', 'FFA', 'GA', 'GWO', 'HHO', 'JAYA', 'MFO', 'MVO', 'PSO', 'SCA', 'SSA', 'WOA']
  Total methods (incl. baseline): 18


## Load Data

In [5]:
# Load features
X_feat = joblib.load(FEATURE_DIR / "X_feat.pkl")
y = np.load(FEATURE_DIR / "y.npy")
subjects = np.load(FEATURE_DIR / "subjects.npy")
le = joblib.load(FEATURE_DIR / "label_encoder.pkl")
print(f"Loaded {len(X_feat)} samples")
print(f"Number of classes: {len(np.unique(y))}")
print(f"Number of subjects: {len(np.unique(subjects))}")
# Get feature dimensions
first_sample = X_feat[0]
FEATURE_DIMS = {
'depth': first_sample['depth_feat'].shape[0],
'sensor': first_sample['sensor_feat'].shape[0],
'skeleton': first_sample['skeleton_feat'].shape[0],
'video': first_sample['video_feat'].shape[0]
}
TOTAL_FEATURES = sum(FEATURE_DIMS.values())
print(f"\nFeature dimensions:")
for modality, dim in FEATURE_DIMS.items():
    print(f"  {modality}: {dim}")
print(f"  TOTAL: {TOTAL_FEATURES}")


Loaded 861 samples
Number of classes: 27
Number of subjects: 8

Feature dimensions:
  depth: 216
  sensor: 652
  skeleton: 1645
  video: 1420
  TOTAL: 3933


## Neural Network Model (Same as Original)

In [6]:
class MultiModalDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.LongTensor(labels)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


class SimpleNN(nn.Module):
    """MLP for unified feature vector (adaptive to feature subset size)"""
    def __init__(self, input_dim, num_classes):
        super().__init__()
        
        # Adaptive hidden layer sizing based on input dimension
        hidden1 = max(128, min(512, input_dim * 2))
        hidden2 = max(64, min(256, hidden1 // 2))
        
        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.BatchNorm1d(hidden1),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(hidden1, hidden2),
            nn.BatchNorm1d(hidden2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden2, num_classes)
        )
    
    def forward(self, x):
        return self.classifier(x)

print("Neural network model defined")


Neural network model defined


## Helper Functions

In [7]:
def prepare_unified_features(X_feat_list, feature_mask=None):
    """Concatenate all modality features into unified vector"""
    unified_features = []
    
    for sample in X_feat_list:
        feat_vector = np.concatenate([
            sample['depth_feat'],
            sample['sensor_feat'],
            sample['skeleton_feat'],
            sample['video_feat']
        ])
        
        if feature_mask is not None:
            feat_vector = feat_vector[feature_mask]
        
        unified_features.append(feat_vector)
    
    return np.array(unified_features)

def calculate_modality_retention(binary_mask, feature_dims):
    """Calculate how many features retained per modality"""
    start_idx = 0
    retention = {}
    
    for modality, dim in feature_dims.items():
        end_idx = start_idx + dim
        modality_mask = binary_mask[start_idx:end_idx]
        num_selected = np.sum(modality_mask)
        percentage = (num_selected / dim) * 100
        
        retention[modality] = {
            'selected': int(num_selected),
            'total': dim,
            'percentage': percentage
        }
        start_idx = end_idx
    
    return retention

def train_and_evaluate(model, train_loader, val_loader, test_loader, num_epochs, lr):
    """Train model and return metrics"""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    model.train()
    for epoch in range(num_epochs):
        for features, labels in train_loader:
            features = features.to(DEVICE)
            labels = labels.to(DEVICE)
            
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
    
    # Evaluate
    model.eval()
    with torch.no_grad():
        # Validation
        val_preds, val_true = [], []
        for features, labels in val_loader:
            features = features.to(DEVICE)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_true.extend(labels.numpy())
        val_acc = accuracy_score(val_true, val_preds)
        
        # Test
        test_preds, test_true = [], []
        for features, labels in test_loader:
            features = features.to(DEVICE)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            test_preds.extend(preds)
            test_true.extend(labels.numpy())
        test_acc = accuracy_score(test_true, test_preds)
    
    return val_acc, test_acc

def count_model_parameters(model):
    """Count trainable parameters in a model"""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def get_model_size_mb(model):
    """Get model size in MB"""
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / (1024 ** 2)

def get_gpu_memory_mb():
    """Get current GPU memory usage in MB"""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / (1024 ** 2)
    return 0.0

def get_dataset_size_mb(X):
    """Get dataset size in MB"""
    return X.nbytes / (1024 ** 2)

print("Helper functions defined")


Helper functions defined


## Enhanced Evaluation (returns full metrics per fold)

In [8]:
def train_and_evaluate_full(model, train_loader, val_loader, test_loader, num_epochs, lr, num_classes):
    """Train model and return comprehensive metrics including predictions for confusion matrix"""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    # Track GPU memory before training
    gpu_mem_before = get_gpu_memory_mb()
    train_start = time.time()

    train_losses = []
    best_val_acc = -1.0
    best_model_state = None
    
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0

        for features, labels in train_loader:
            features = features.to(DEVICE)
            labels = labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        train_losses.append(epoch_loss / len(train_loader))

        # Check validation accuracy after each epoch
        model.eval()
        val_correct, val_total = 0, 0
        with torch.no_grad():
            for features, labels in val_loader:
                features = features.to(DEVICE)
                outputs = model(features)
                preds = torch.argmax(outputs, dim=1)
                val_correct += (preds.cpu() == labels).sum().item()
                val_total += labels.size(0)

        epoch_val_acc = val_correct / val_total

        if epoch_val_acc > best_val_acc:
            best_val_acc = epoch_val_acc
            best_model_state = deepcopy(model.state_dict())
    
    train_time = time.time() - train_start
    gpu_mem_after = get_gpu_memory_mb()

    # Restore best model
    model.load_state_dict(best_model_state)
    
    # Final evaluation
    model.eval()
    with torch.no_grad():
        val_preds, val_true = [], []
        for features, labels in val_loader:
            features = features.to(DEVICE)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_true.extend(labels.numpy())
        
        # Test
        test_preds, test_true = [], []
        for features, labels in test_loader:
            features = features.to(DEVICE)
            outputs = model(features)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            test_preds.extend(preds)
            test_true.extend(labels.numpy())
    
    val_acc = accuracy_score(val_true, val_preds)
    test_acc = accuracy_score(test_true, test_preds)
    
    metrics = {
        'val_acc': val_acc,
        'test_acc': test_acc,
        'test_f1_macro': f1_score(test_true, test_preds, average='macro', zero_division=0),
        'test_f1_weighted': f1_score(test_true, test_preds, average='weighted', zero_division=0),
        'test_precision_macro': precision_score(test_true, test_preds, average='macro', zero_division=0),
        'test_recall_macro': recall_score(test_true, test_preds, average='macro', zero_division=0),
        'test_preds': np.array(test_preds),
        'test_true': np.array(test_true),
        'val_preds': np.array(val_preds),
        'val_true': np.array(val_true),
        'train_time_sec': train_time,
        'gpu_mem_before_mb': gpu_mem_before,
        'gpu_mem_after_mb': gpu_mem_after,
        'gpu_mem_peak_mb': torch.cuda.max_memory_allocated() / (1024**2) if torch.cuda.is_available() else 0,
        'model_params': count_model_parameters(model),
        'model_size_mb': get_model_size_mb(model),
        'train_losses': train_losses,
    }
    return metrics

print("Enhanced evaluation function defined")


Enhanced evaluation function defined


## Feature Selection Methods\n### 0. EvoloPy Metaheuristics

In [9]:
# ============================================================================
# EXACT EVOLOPY BAT IMPLEMENTATION FROM ORIGINAL CODE
# ============================================================================

def train_model_quick(model, train_loader, val_loader, epochs, lr, device):
    """Quick training for fitness evaluation"""
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    best_val_loss = float('inf')
    best_val_acc = 0.0
    
    for epoch in range(epochs):
        model.train()
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(features), labels)
            loss.backward()
            optimizer.step()
        
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for features, labels in val_loader:
                features, labels = features.to(device), labels.to(device)
                outputs = model(features)
                val_loss += criterion(outputs, labels).item()
                val_correct += (torch.argmax(outputs, 1) == labels).sum().item()
                val_total += labels.size(0)
        
        val_loss /= len(val_loader)
        val_acc = val_correct / val_total
        if val_loss < best_val_loss:
            best_val_loss, best_val_acc = val_loss, val_acc
    
    return best_val_loss, best_val_acc

# ============================================================================
# FITNESS FUNCTION: Using (1 - accuracy)
# ============================================================================
# Rationale: (1 - accuracy) is preferred over loss because:
#   - It directly optimizes the metric we care about (accuracy)
#   - It is bounded in [0, 1], making it cleaner for metaheuristic optimization
#   - Loss can vary in scale across different feature subsets
#   - Accuracy-based fitness is more interpretable and comparable across methods
#   - Less sensitive to calibration issues of the neural network

def create_fitness_function_evolopy(X_train, y_train, X_val, y_val, num_classes):
    """Create fitness function for EvoloPy using (1 - accuracy) + feature penalty"""
    eval_count = [0]  # track evaluations
    
    def fitness_function(binary_mask):
        try:
            if binary_mask.dtype != bool:
                binary_mask = binary_mask > 0.5

            num_selected = np.sum(binary_mask)
            if num_selected == 0:
                return 1.0
            
            X_tr_sel = X_train[:, binary_mask]
            X_val_sel = X_val[:, binary_mask]
            
            train_dataset = MultiModalDataset(X_tr_sel, y_train)
            val_dataset = MultiModalDataset(X_val_sel, y_val)
            train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
            
            model = SimpleNN(X_tr_sel.shape[1], num_classes).to(DEVICE)
            
            val_loss, val_acc = train_model_quick(
                model, train_loader, val_loader,
                epochs=30, lr=1e-3, device=DEVICE
            )
            
            del model, train_dataset, val_dataset, train_loader, val_loader
            torch.cuda.empty_cache()
            
            eval_count[0] += 1
            
            # Weighted fitness: accuracy + feature reduction pressure
            alpha = 0.95  # weight for accuracy
            beta = 0.05   # weight for feature reduction
            feature_ratio = num_selected / len(binary_mask)

            fitness = alpha * (1.0 - val_acc) + beta * feature_ratio
            return fitness
            
        except Exception as e:
            print(f"Error in fitness: {e}")
            return 1.0
        
    return fitness_function


def s_transfer(x):
    return 1 / (1 + np.exp(-10 * (x - 0.5)))


def run_evolopy_optimizer(optimizer_name, optimizer_func, X_train, y_train, X_val, y_val, num_classes):
    """Run any EvoloPy optimizer generically"""
    print(f"    Running EvoloPy {optimizer_name}...")
    print(f"      Fitness: (1 - accuracy), Pop: {N_POPULATION}, Iter: {MAX_ITERATIONS}")
    
    fitness_func = create_fitness_function_evolopy(X_train, y_train, X_val, y_val, num_classes)
    
    start = time.time()
    solution = optimizer_func(fitness_func, 0, 1, TOTAL_FEATURES, N_POPULATION, MAX_ITERATIONS)
    exec_time = time.time() - start
    
    # binary_mask = solution.bestIndividual > 0.5
    binary_mask = np.random.rand(TOTAL_FEATURES) < s_transfer(solution.bestIndividual)
    modality_ret = calculate_modality_retention(binary_mask, FEATURE_DIMS)
    
    results = {
        'mask': binary_mask,
        'convergence': solution.convergence.tolist() if hasattr(solution.convergence, 'tolist') else list(solution.convergence),
        'best_fitness': float(solution.convergence[-1]) if len(solution.convergence) > 0 else float('inf'),
        'execution_time': exec_time,
        'num_selected': int(np.sum(binary_mask)),
        'num_total': len(binary_mask),
        'modality_retention': modality_ret,
        'method': f'meta_{optimizer_name}'
    }
    
    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), "
          f"Fitness: {results['best_fitness']:.4f}, Time: {exec_time:.1f}s")
    print(f"      Modality: D={modality_ret['depth']['percentage']:.0f}%, "
          f"S={modality_ret['sensor']['percentage']:.0f}%, "
          f"Sk={modality_ret['skeleton']['percentage']:.0f}%, "
          f"V={modality_ret['video']['percentage']:.0f}%")
    
    return results

print("All EvoloPy metaheuristic runners defined")
print(f"Available optimizers: {list(EVOLOPY_OPTIMIZERS.keys())}")


All EvoloPy metaheuristic runners defined
Available optimizers: ['BAT', 'CS', 'DE', 'FFA', 'GA', 'GWO', 'HHO', 'JAYA', 'MFO', 'MVO', 'PSO', 'SCA', 'SSA', 'WOA']


### 1-3. Standard Feature Selection Methods (Mutual Info, RFE, LASSO)

In [10]:
def run_mutual_info_fs(X_train, y_train, X_val, y_val, num_classes):
    """Run Mutual Information feature selection"""
    print("    Running Mutual Information...")
    
    start_time = time.time()
    
    # Calculate mutual information scores
    mi_scores = mutual_info_classif(X_train, y_train, random_state=42)
    
    # Select top k features (target percentage)
    k = int(TOTAL_FEATURES * TARGET_FEATURE_PERCENTAGE)
    top_k_indices = np.argsort(mi_scores)[::-1][:k]
    
    binary_mask = np.zeros(TOTAL_FEATURES, dtype=bool)
    binary_mask[top_k_indices] = True
    
    execution_time = time.time() - start_time
    
    modality_retention = calculate_modality_retention(binary_mask, FEATURE_DIMS)
    
    results = {
        'mask': binary_mask,
        'execution_time': execution_time,
        'num_selected': int(np.sum(binary_mask)),
        'num_total': len(binary_mask),
        'modality_retention': modality_retention,
        'mi_scores': mi_scores,
        'method': 'mutual_info'
    }
    
    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), "
          f"Time: {results['execution_time']:.1f}s")
    
    return results


def run_rfe_fs(X_train, y_train, X_val, y_val, num_classes):
    """Run RFE feature selection"""
    print("    Running RFE...")
    
    start_time = time.time()
    
    # Use Random Forest as base estimator
    estimator = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
    
    # Select top k features
    k = int(TOTAL_FEATURES * TARGET_FEATURE_PERCENTAGE)
    selector = RFE(estimator, n_features_to_select=k, step=50)  # Remove 50 features at a time
    selector.fit(X_train, y_train)
    
    binary_mask = selector.support_
    
    execution_time = time.time() - start_time
    
    modality_retention = calculate_modality_retention(binary_mask, FEATURE_DIMS)
    
    results = {
        'mask': binary_mask,
        'execution_time': execution_time,
        'num_selected': int(np.sum(binary_mask)),
        'num_total': len(binary_mask),
        'modality_retention': modality_retention,
        'ranking': selector.ranking_,
        'method': 'rfe'
    }
    
    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), "
          f"Time: {results['execution_time']:.1f}s")
    
    return results


def run_lasso_fs(X_train, y_train, X_val, y_val, num_classes):
    """Run LASSO feature selection"""
    print("    Running LASSO...")
    start_time = time.time()

    # Train Lasso to get feature importance scores
    lasso = LogisticRegression(
        penalty='l1',
        C=0.01,
        solver='liblinear',
        random_state=42,
        max_iter=1000
    )
    lasso.fit(X_train, y_train)

    # Rank features by coefficient magnitude across all classes
    coef_abs = np.abs(lasso.coef_).sum(axis=0)

    # Select top-k features to match target percentage
    target_count = int(TOTAL_FEATURES * TARGET_FEATURE_PERCENTAGE)
    top_indices = np.argsort(coef_abs)[::-1][:target_count]
    binary_mask = np.zeros(TOTAL_FEATURES, dtype=bool)
    binary_mask[top_indices] = True

    execution_time = time.time() - start_time
    modality_retention = calculate_modality_retention(binary_mask, FEATURE_DIMS)

    results = {
        'mask': binary_mask,
        'execution_time': execution_time,
        'num_selected': int(np.sum(binary_mask)),
        'num_total': len(binary_mask),
        'modality_retention': modality_retention,
        'coefficients': coef_abs,
        'method': 'lasso'
    }

    print(f"      Selected: {results['num_selected']}/{results['num_total']} "
          f"({results['num_selected']/results['num_total']*100:.1f}%), "
          f"Time: {results['execution_time']:.1f}s")

    return results

print("Standard FS methods defined (Mutual Info, RFE, LASSO)")


Standard FS methods defined (Mutual Info, RFE, LASSO)


## Main Per-Fold Experiment Runner

In [11]:
def run_all_methods_per_fold():
    """
    Run ALL methods across ALL folds, saving comprehensive per-fold results.
    Each fold is treated as an independent dataset version (no averaging).
    """
    print("="*80)
    print("STARTING COMPREHENSIVE PER-FOLD EXPERIMENTS")
    print(f"Methods: baseline + 3 standard + {len(EVOLOPY_OPTIMIZERS)} metaheuristics = {len(ALL_METHODS)} total")
    print(f"Folds: {N_FOLDS}")
    print("="*80)
    
    # Prepare unified features
    X_unified = prepare_unified_features(X_feat)
    num_classes = len(np.unique(y))
    
    # Master results dict: {method_name: {fold_idx: {...}}}
    master_results = {m: {} for m in ALL_METHODS}
    
    # LOSO cross-validation
    gkf = GroupKFold(n_splits=N_FOLDS)
    
    for fold_idx, (train_val_idx, test_idx) in enumerate(gkf.split(X_unified, y, groups=subjects)):
        print(f"\n{'#'*80}")
        print(f"# FOLD {fold_idx + 1}/{N_FOLDS}")
        print(f"{'#'*80}")
        
        # Split data
        X_train_val = X_unified[train_val_idx]
        y_train_val = y[train_val_idx]
        X_test = X_unified[test_idx]
        y_test = y[test_idx]
        
        # Further split train_val into train and val
        n_val = len(train_val_idx) // 5
        val_indices = np.random.choice(len(train_val_idx), n_val, replace=False)
        train_indices = np.array([i for i in range(len(train_val_idx)) if i not in val_indices])
        
        X_train = X_train_val[train_indices]
        y_train = y_train_val[train_indices]
        X_val = X_train_val[val_indices]
        y_val = y_train_val[val_indices]
        
        print(f"  Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")
        
        # Normalize
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_val = scaler.transform(X_val)
        X_test = scaler.transform(X_test)
        
        # Original dataset size
        orig_dataset_size_mb = get_dataset_size_mb(X_train) + get_dataset_size_mb(X_val) + get_dataset_size_mb(X_test)
        
        # =====================================================================
        # Run each method
        # =====================================================================
        for method_name in ALL_METHODS:
            print(f"\n  --- {method_name.upper()} ---")
            
            fs_start_time = time.time()
            
            # Feature selection
            if method_name == 'baseline':
                feature_mask = np.ones(TOTAL_FEATURES, dtype=bool)
                fs_results = {
                    'mask': feature_mask,
                    'execution_time': 0,
                    'num_selected': TOTAL_FEATURES,
                    'num_total': TOTAL_FEATURES,
                    'method': 'baseline'
                }
            elif method_name == 'mutual_info':
                fs_results = run_mutual_info_fs(X_train, y_train, X_val, y_val, num_classes)
                feature_mask = fs_results['mask']
            elif method_name == 'rfe':
                fs_results = run_rfe_fs(X_train, y_train, X_val, y_val, num_classes)
                feature_mask = fs_results['mask']
            elif method_name == 'lasso':
                fs_results = run_lasso_fs(X_train, y_train, X_val, y_val, num_classes)
                feature_mask = fs_results['mask']
            elif method_name.startswith('meta_'):
                opt_name = method_name.replace('meta_', '')
                opt_func = EVOLOPY_OPTIMIZERS[opt_name]
                fs_results = run_evolopy_optimizer(opt_name, opt_func, X_train, y_train, X_val, y_val, num_classes)
                feature_mask = fs_results['mask']
            else:
                raise ValueError(f"Unknown method: {method_name}")
            
            fs_time = time.time() - fs_start_time
            
            # Apply feature mask
            X_train_sel = X_train[:, feature_mask]
            X_val_sel = X_val[:, feature_mask]
            X_test_sel = X_test[:, feature_mask]
            
            # Dataset size after selection
            opt_dataset_size_mb = get_dataset_size_mb(X_train_sel) + get_dataset_size_mb(X_val_sel) + get_dataset_size_mb(X_test_sel)
            
            # Build and train final model
            model = SimpleNN(X_train_sel.shape[1], num_classes).to(DEVICE)
            
            train_dataset = MultiModalDataset(X_train_sel, y_train)
            val_dataset = MultiModalDataset(X_val_sel, y_val)
            test_dataset = MultiModalDataset(X_test_sel, y_test)
            
            train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
            val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)
            test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)
            
            if torch.cuda.is_available():
                torch.cuda.reset_peak_memory_stats()
            
            eval_metrics = train_and_evaluate_full(
                model, train_loader, val_loader, test_loader, N_EPOCHS, LEARNING_RATE, num_classes
            )
            
            # Modality retention
            if method_name != 'baseline':
                modality_ret = fs_results.get('modality_retention', calculate_modality_retention(feature_mask, FEATURE_DIMS))
            else:
                modality_ret = {m: {'selected': d, 'total': d, 'percentage': 100.0} for m, d in FEATURE_DIMS.items()}
            
            # Compile comprehensive fold result
            fold_result = {
                'fold': fold_idx,
                'method': method_name,
                'num_train': len(X_train),
                'num_val': len(X_val),
                'num_test': len(X_test),
                # Accuracy & classification metrics
                'val_acc': eval_metrics['val_acc'],
                'test_acc': eval_metrics['test_acc'],
                'test_f1_macro': eval_metrics['test_f1_macro'],
                'test_f1_weighted': eval_metrics['test_f1_weighted'],
                'test_precision_macro': eval_metrics['test_precision_macro'],
                'test_recall_macro': eval_metrics['test_recall_macro'],
                # Feature selection info
                'num_features_selected': int(np.sum(feature_mask)),
                'num_features_total': TOTAL_FEATURES,
                'feature_retention_pct': float(np.sum(feature_mask) / TOTAL_FEATURES * 100),
                'modality_retention': modality_ret,
                'feature_mask': feature_mask,
                # Timing
                'fs_execution_time': fs_results.get('execution_time', 0),
                'train_time_sec': eval_metrics['train_time_sec'],
                'total_time_sec': fs_time + eval_metrics['train_time_sec'],
                # Model size
                'model_params': eval_metrics['model_params'],
                'model_size_mb': eval_metrics['model_size_mb'],
                # Dataset size
                'original_dataset_size_mb': orig_dataset_size_mb,
                'optimized_dataset_size_mb': opt_dataset_size_mb,
                'dataset_reduction_pct': float((1 - opt_dataset_size_mb / orig_dataset_size_mb) * 100) if orig_dataset_size_mb > 0 else 0,
                # GPU
                'gpu_mem_peak_mb': eval_metrics['gpu_mem_peak_mb'],
                # Convergence (metaheuristics only)
                'convergence': fs_results.get('convergence', None),
                'best_fitness': fs_results.get('best_fitness', None),
            }
            
            master_results[method_name][fold_idx] = fold_result
            
            # Save per-fold result immediately
            fold_dir = RESULTS_ROOT / method_name
            fold_save = {k: v for k, v in fold_result.items() if k not in ['feature_mask']}
            # Convert numpy to python types for JSON
            fold_save_clean = {}
            for k, v in fold_save.items():
                if isinstance(v, np.ndarray):
                    fold_save_clean[k] = v.tolist()
                elif isinstance(v, (np.floating, np.integer)):
                    fold_save_clean[k] = float(v)
                elif isinstance(v, dict):
                    fold_save_clean[k] = {}
                    for kk, vv in v.items():
                        if isinstance(vv, dict):
                            fold_save_clean[k][kk] = {kkk: float(vvv) if isinstance(vvv, (np.floating, np.integer)) else vvv for kkk, vvv in vv.items()}
                        else:
                            fold_save_clean[k][kk] = float(vv) if isinstance(vv, (np.floating, np.integer)) else vv
                else:
                    fold_save_clean[k] = v
            
            with open(fold_dir / f"fold_{fold_idx+1}.json", 'w') as f:
                json_lib.dump(fold_save_clean, f, indent=2, default=str)
            
            np.save(fold_dir / f"fold_{fold_idx+1}_mask.npy", feature_mask)
            
            print(f"    Test Acc: {eval_metrics['test_acc']*100:.2f}%, "
                  f"F1: {eval_metrics['test_f1_macro']*100:.2f}%, "
                  f"Features: {np.sum(feature_mask)}/{TOTAL_FEATURES} "
                  f"({np.sum(feature_mask)/TOTAL_FEATURES*100:.1f}%), "
                  f"Model: {eval_metrics['model_size_mb']:.3f}MB")
            
            # Cleanup
            del model, train_dataset, val_dataset, test_dataset
            del train_loader, val_loader, test_loader
            torch.cuda.empty_cache()
            gc.collect()
    
    print(f"\n{'='*80}")
    print("ALL PER-FOLD EXPERIMENTS COMPLETED")
    print(f"{'='*80}")
    
    return master_results

print("Per-fold experiment runner defined")


Per-fold experiment runner defined


## Run All Experiments

In [12]:
master_results = run_all_methods_per_fold()

STARTING COMPREHENSIVE PER-FOLD EXPERIMENTS
Methods: baseline + 3 standard + 14 metaheuristics = 18 total
Folds: 8

################################################################################
# FOLD 1/8
################################################################################
  Train: 603, Val: 150, Test: 108

  --- BASELINE ---
    Test Acc: 100.00%, F1: 100.00%, Features: 3933/3933 (100.0%), Model: 8.223MB

  --- MUTUAL_INFO ---
    Running Mutual Information...
      Selected: 1966/3933 (50.0%), Time: 21.0s
    Test Acc: 94.44%, F1: 94.07%, Features: 1966/3933 (50.0%), Model: 4.381MB

  --- RFE ---
    Running RFE...
      Selected: 1966/3933 (50.0%), Time: 4.0s
    Test Acc: 99.07%, F1: 99.06%, Features: 1966/3933 (50.0%), Model: 4.381MB

  --- LASSO ---
    Running LASSO...
      Selected: 1966/3933 (50.0%), Time: 1.1s
    Test Acc: 82.41%, F1: 81.60%, Features: 1966/3933 (50.0%), Model: 4.381MB

  --- META_BAT ---
    Running EvoloPy BAT...
      Fitness: (1 - accurac

## Build Comprehensive Per-Fold Results CSV

In [13]:
# Build a single massive DataFrame with every fold x method combination
rows = []

for method_name in ALL_METHODS:
    for fold_idx in range(N_FOLDS):
        if fold_idx not in master_results[method_name]:
            continue
        r = master_results[method_name][fold_idx]
        
        row = {
            'Method': method_name,
            'Fold': fold_idx + 1,
            'Test Accuracy (%)': r['test_acc'] * 100,
            'Val Accuracy (%)': r['val_acc'] * 100,
            'F1 Macro (%)': r['test_f1_macro'] * 100,
            'F1 Weighted (%)': r['test_f1_weighted'] * 100,
            'Precision Macro (%)': r['test_precision_macro'] * 100,
            'Recall Macro (%)': r['test_recall_macro'] * 100,
            'Features Selected': r['num_features_selected'],
            'Features Total': r['num_features_total'],
            'Feature Retention (%)': r['feature_retention_pct'],
            'FS Time (s)': r['fs_execution_time'],
            'Train Time (s)': r['train_time_sec'],
            'Total Time (s)': r['total_time_sec'],
            'Model Params': r['model_params'],
            'Model Size (MB)': r['model_size_mb'],
            'Original Dataset (MB)': r['original_dataset_size_mb'],
            'Optimized Dataset (MB)': r['optimized_dataset_size_mb'],
            'Dataset Reduction (%)': r['dataset_reduction_pct'],
            'GPU Peak (MB)': r['gpu_mem_peak_mb'],
        }
        
        # Add per-modality retention
        for mod_name in FEATURE_DIMS.keys():
            mod_ret = r['modality_retention'].get(mod_name, {})
            if isinstance(mod_ret, dict):
                row[f'{mod_name}_retention (%)'] = mod_ret.get('percentage', 0)
                row[f'{mod_name}_selected'] = mod_ret.get('selected', 0)
            else:
                row[f'{mod_name}_retention (%)'] = float(mod_ret)
        
        rows.append(row)

df_all = pd.DataFrame(rows)
df_all = df_all.round(4)

# Save
csv_path = RESULTS_ROOT / "all_results_per_fold.csv"
df_all.to_csv(csv_path, index=False)
print(f"Saved comprehensive per-fold CSV: {csv_path}")
print(f"Shape: {df_all.shape}")
print(df_all.head(20).to_string())


Saved comprehensive per-fold CSV: results_per_fold_new/all_results_per_fold.csv
Shape: (144, 28)
         Method  Fold  Test Accuracy (%)  Val Accuracy (%)  F1 Macro (%)  F1 Weighted (%)  Precision Macro (%)  Recall Macro (%)  Features Selected  Features Total  Feature Retention (%)  FS Time (s)  Train Time (s)  Total Time (s)  Model Params  Model Size (MB)  Original Dataset (MB)  Optimized Dataset (MB)  Dataset Reduction (%)  GPU Peak (MB)  depth_retention (%)  depth_selected  sensor_retention (%)  sensor_selected  skeleton_retention (%)  skeleton_selected  video_retention (%)  video_selected
0      baseline     1           100.0000             100.0      100.0000         100.0000             100.0000          100.0000               3933            3933               100.0000       0.0000          8.3952          8.3953       2154011           8.2228                25.8355                 25.8355                 0.0000        67.0527             100.0000             216              1

## Per-Method Summary Table (with per-fold detail)

In [14]:
# Also create a summary table (mean/std across folds)
summary_rows = []
for method_name in ALL_METHODS:
    method_df = df_all[df_all['Method'] == method_name]
    if len(method_df) == 0:
        continue
    
    row = {
        'Method': method_name,
        'Mean Test Acc (%)': method_df['Test Accuracy (%)'].mean(),
        'Std Test Acc (%)': method_df['Test Accuracy (%)'].std(),
        'Mean F1 Macro (%)': method_df['F1 Macro (%)'].mean(),
        'Std F1 Macro (%)': method_df['F1 Macro (%)'].std(),
        'Mean Features Selected': method_df['Features Selected'].mean(),
        'Mean Feature Retention (%)': method_df['Feature Retention (%)'].mean(),
        'Mean FS Time (s)': method_df['FS Time (s)'].mean(),
        'Mean Train Time (s)': method_df['Train Time (s)'].mean(),
        'Mean Total Time (s)': method_df['Total Time (s)'].mean(),
        'Mean Model Params': method_df['Model Params'].mean(),
        'Mean Model Size (MB)': method_df['Model Size (MB)'].mean(),
        'Mean Dataset Reduction (%)': method_df['Dataset Reduction (%)'].mean(),
        'Mean GPU Peak (MB)': method_df['GPU Peak (MB)'].mean(),
    }
    summary_rows.append(row)

df_summary = pd.DataFrame(summary_rows).round(2)
df_summary.to_csv(RESULTS_ROOT / "summary_all_methods.csv", index=False)
print("\nSUMMARY TABLE (mean across folds):")
print(df_summary.to_string(index=False))



SUMMARY TABLE (mean across folds):
     Method  Mean Test Acc (%)  Std Test Acc (%)  Mean F1 Macro (%)  Std F1 Macro (%)  Mean Features Selected  Mean Feature Retention (%)  Mean FS Time (s)  Mean Train Time (s)  Mean Total Time (s)  Mean Model Params  Mean Model Size (MB)  Mean Dataset Reduction (%)  Mean GPU Peak (MB)
   baseline              93.39              4.68              92.66              5.23                 3933.00                      100.00              0.00                 8.09                 8.09          2154011.0                  8.22                        0.00               67.05
mutual_info              93.97              2.97              92.92              3.49                 1966.00                       49.99             20.87                 6.39                27.26          1146907.0                  4.38                       50.01               44.47
        rfe              94.55              4.73              93.52              5.71                 1

## Comprehensive Visualizations

In [15]:
# ============================================================================
# PLOT 1: Test Accuracy per fold for all methods (heatmap)
# ============================================================================
pivot = df_all.pivot_table(values='Test Accuracy (%)', index='Method', columns='Fold')
pivot = pivot.reindex(ALL_METHODS)

fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlGnBu', ax=ax, linewidths=0.5, cbar_kws={'label': 'Test Accuracy (%)'})
ax.set_title('Test Accuracy (%) per Method per Fold', fontsize=16)
ax.set_ylabel('Method')
ax.set_xlabel('Fold')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '01_accuracy_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 01_accuracy_heatmap.png")


Saved: 01_accuracy_heatmap.png


In [16]:
# ============================================================================
# PLOT 2: Box plot of test accuracy across folds for each method
# ============================================================================
fig, ax = plt.subplots(figsize=(16, 7))
methods_sorted = df_summary.sort_values('Mean Test Acc (%)', ascending=False)['Method'].tolist()
order = methods_sorted

sns.boxplot(data=df_all, x='Method', y='Test Accuracy (%)', order=order, ax=ax, palette='Set2')
sns.stripplot(data=df_all, x='Method', y='Test Accuracy (%)', order=order, ax=ax, 
              color='black', alpha=0.5, size=4, jitter=True)
ax.set_title('Test Accuracy Distribution Across Folds', fontsize=16)
ax.set_xlabel('')
plt.xticks(rotation=60, ha='right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '02_accuracy_boxplot.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 02_accuracy_boxplot.png")


Saved: 02_accuracy_boxplot.png


In [17]:
# ============================================================================
# PLOT 3: Feature Retention per method (bar chart with error bars)
# ============================================================================
fig, ax = plt.subplots(figsize=(16, 7))
feat_summary = df_all.groupby('Method')['Feature Retention (%)'].agg(['mean', 'std']).reindex(ALL_METHODS)
bars = ax.bar(feat_summary.index, feat_summary['mean'], yerr=feat_summary['std'], capsize=3, alpha=0.7, color='coral')
ax.axhline(y=100, color='r', linestyle='--', label='All Features (100%)')
ax.set_ylabel('Feature Retention (%)', fontsize=12)
ax.set_title('Feature Retention by Method', fontsize=16)
ax.legend()
plt.xticks(rotation=60, ha='right')
ax.grid(axis='y', alpha=0.3)

# Value labels
for bar, mean_val in zip(bars, feat_summary['mean']):
    if not np.isnan(mean_val):
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 2,
                f'{mean_val:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
plt.savefig(PLOTS_DIR / '03_feature_retention.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 03_feature_retention.png")


Saved: 03_feature_retention.png


In [18]:
# ============================================================================
# PLOT 4: Accuracy vs Feature Retention trade-off (scatter)
# ============================================================================
fig, ax = plt.subplots(figsize=(12, 8))

for method in ALL_METHODS:
    mdf = df_all[df_all['Method'] == method]
    if len(mdf) == 0:
        continue
    ax.scatter(mdf['Feature Retention (%)'], mdf['Test Accuracy (%)'], 
               label=method, s=60, alpha=0.7)
    # Draw mean point larger
    ax.scatter(mdf['Feature Retention (%)'].mean(), mdf['Test Accuracy (%)'].mean(),
               s=150, marker='X', edgecolors='black', linewidths=1.5)

ax.set_xlabel('Feature Retention (%)', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Accuracy vs Feature Retention Trade-off', fontsize=16)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '04_accuracy_vs_retention.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 04_accuracy_vs_retention.png")


Saved: 04_accuracy_vs_retention.png


In [19]:
# ============================================================================
# PLOT 5: Per-modality retention heatmap (average across folds)
# ============================================================================
modality_data = {}
for method in ALL_METHODS:
    modality_data[method] = {}
    for mod in FEATURE_DIMS.keys():
        col = f'{mod}_retention (%)'
        if col in df_all.columns:
            mdf = df_all[df_all['Method'] == method]
            modality_data[method][mod] = mdf[col].mean() if len(mdf) > 0 else 0

mod_df = pd.DataFrame(modality_data).T
mod_df = mod_df.reindex(ALL_METHODS)

fig, ax = plt.subplots(figsize=(10, 10))
sns.heatmap(mod_df, annot=True, fmt='.1f', cmap='RdYlGn', ax=ax, linewidths=0.5,
            vmin=0, vmax=100, cbar_kws={'label': 'Retention (%)'})
ax.set_title('Average Modality-wise Feature Retention (%)', fontsize=14)
ax.set_ylabel('Method')
ax.set_xlabel('Modality')
plt.tight_layout()
plt.savefig(PLOTS_DIR / '05_modality_retention_heatmap.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 05_modality_retention_heatmap.png")


Saved: 05_modality_retention_heatmap.png


In [20]:
# ============================================================================
# PLOT 6: Execution time comparison
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# FS time
time_summary = df_all.groupby('Method')['FS Time (s)'].agg(['mean', 'std']).reindex(ALL_METHODS)
bars = axes[0].bar(time_summary.index, time_summary['mean'], yerr=time_summary['std'], capsize=3, alpha=0.7, color='steelblue')
axes[0].set_ylabel('Feature Selection Time (s)', fontsize=12)
axes[0].set_title('Feature Selection Time', fontsize=14)
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=60, ha='right')
axes[0].grid(axis='y', alpha=0.3)

# Total time
total_summary = df_all.groupby('Method')['Total Time (s)'].agg(['mean', 'std']).reindex(ALL_METHODS)
bars2 = axes[1].bar(total_summary.index, total_summary['mean'], yerr=total_summary['std'], capsize=3, alpha=0.7, color='darkorange')
axes[1].set_ylabel('Total Time (s)', fontsize=12)
axes[1].set_title('Total Time (FS + Training)', fontsize=14)
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=60, ha='right')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / '06_execution_time.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 06_execution_time.png")


Saved: 06_execution_time.png


In [21]:
# ============================================================================
# PLOT 7: Model size & dataset size comparison
# ============================================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Model params
param_summary = df_all.groupby('Method')['Model Params'].mean().reindex(ALL_METHODS)
axes[0].bar(param_summary.index, param_summary.values, alpha=0.7, color='mediumpurple')
axes[0].set_ylabel('Model Parameters', fontsize=12)
axes[0].set_title('Average Model Parameters', fontsize=14)
plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=60, ha='right')
axes[0].grid(axis='y', alpha=0.3)

# Dataset reduction
ds_summary = df_all.groupby('Method')['Dataset Reduction (%)'].mean().reindex(ALL_METHODS)
axes[1].bar(ds_summary.index, ds_summary.values, alpha=0.7, color='seagreen')
axes[1].set_ylabel('Dataset Size Reduction (%)', fontsize=12)
axes[1].set_title('Average Dataset Size Reduction', fontsize=14)
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=60, ha='right')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / '07_model_dataset_size.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 07_model_dataset_size.png")


Saved: 07_model_dataset_size.png


In [22]:
# ============================================================================
# PLOT 8: Metaheuristics-only comparison (grouped bar: accuracy & retention)
# ============================================================================
meta_methods = [m for m in ALL_METHODS if m.startswith('meta_')]

if len(meta_methods) > 0:
    meta_df = df_all[df_all['Method'].isin(meta_methods)]
    
    fig, axes = plt.subplots(2, 1, figsize=(16, 12))
    
    # Accuracy
    meta_acc = meta_df.groupby('Method')['Test Accuracy (%)'].agg(['mean', 'std']).reindex(meta_methods)
    bars = axes[0].bar(range(len(meta_methods)), meta_acc['mean'], yerr=meta_acc['std'], 
                        capsize=4, alpha=0.7, color=plt.cm.tab20(np.linspace(0, 1, len(meta_methods))))
    axes[0].set_xticks(range(len(meta_methods)))
    axes[0].set_xticklabels([m.replace('meta_', '') for m in meta_methods], rotation=45, ha='right')
    axes[0].set_ylabel('Test Accuracy (%)')
    axes[0].set_title('Metaheuristics: Test Accuracy Comparison', fontsize=14)
    axes[0].grid(axis='y', alpha=0.3)
    
    for i, (bar, val) in enumerate(zip(bars, meta_acc['mean'])):
        if not np.isnan(val):
            axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + meta_acc['std'].iloc[i] + 0.5,
                        f'{val:.1f}', ha='center', va='bottom', fontsize=8)
    
    # Feature retention
    meta_feat = meta_df.groupby('Method')['Feature Retention (%)'].agg(['mean', 'std']).reindex(meta_methods)
    bars2 = axes[1].bar(range(len(meta_methods)), meta_feat['mean'], yerr=meta_feat['std'],
                         capsize=4, alpha=0.7, color=plt.cm.tab20(np.linspace(0, 1, len(meta_methods))))
    axes[1].set_xticks(range(len(meta_methods)))
    axes[1].set_xticklabels([m.replace('meta_', '') for m in meta_methods], rotation=45, ha='right')
    axes[1].set_ylabel('Feature Retention (%)')
    axes[1].set_title('Metaheuristics: Feature Retention Comparison', fontsize=14)
    axes[1].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / '08_metaheuristics_comparison.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("Saved: 08_metaheuristics_comparison.png")


Saved: 08_metaheuristics_comparison.png


In [23]:
# ============================================================================
# PLOT 9: Convergence curves for metaheuristics (average across folds)
# ============================================================================
meta_methods = [m for m in ALL_METHODS if m.startswith('meta_')]

if len(meta_methods) > 0:
    fig, ax = plt.subplots(figsize=(14, 8))
    
    for method in meta_methods:
        all_conv = []
        for fold_idx in range(N_FOLDS):
            if fold_idx in master_results[method]:
                conv = master_results[method][fold_idx].get('convergence', None)
                if conv is not None:
                    all_conv.append(conv)
        
        if all_conv:
            # Pad to same length
            max_len = max(len(c) for c in all_conv)
            padded = []
            for c in all_conv:
                if len(c) < max_len:
                    c = list(c) + [c[-1]] * (max_len - len(c))
                padded.append(c)
            
            avg_conv = np.mean(padded, axis=0)
            ax.plot(avg_conv, label=method.replace('meta_', ''), linewidth=1.5)
    
    ax.set_xlabel('Iteration', fontsize=12)
    ax.set_ylabel('Fitness (1 - Accuracy)', fontsize=12)
    ax.set_title('Metaheuristic Convergence Curves (averaged across folds)', fontsize=14)
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(PLOTS_DIR / '09_convergence_curves.png', dpi=300, bbox_inches='tight')
    plt.close()
    print("Saved: 09_convergence_curves.png")


Saved: 09_convergence_curves.png


In [24]:
# ============================================================================
# PLOT 10: Feature importance - which features most commonly retained/removed
# ============================================================================
# Count how many times each feature index was selected across all methods and folds
feature_freq = np.zeros(TOTAL_FEATURES)
feature_freq_per_method = {}

for method in ALL_METHODS:
    if method == 'baseline':
        continue
    method_freq = np.zeros(TOTAL_FEATURES)
    count = 0
    for fold_idx in range(N_FOLDS):
        if fold_idx in master_results[method]:
            mask = master_results[method][fold_idx]['feature_mask']
            feature_freq += mask.astype(float)
            method_freq += mask.astype(float)
            count += 1
    if count > 0:
        feature_freq_per_method[method] = method_freq / count

# Normalize overall
total_runs = (len(ALL_METHODS) - 1) * N_FOLDS  # exclude baseline
feature_freq_normalized = feature_freq / max(total_runs, 1)

# Plot feature selection frequency
fig, axes = plt.subplots(2, 1, figsize=(18, 10))

# Overall
axes[0].bar(range(TOTAL_FEATURES), feature_freq_normalized, width=1.0, alpha=0.7, color='steelblue')
axes[0].set_xlabel('Feature Index')
axes[0].set_ylabel('Selection Frequency')
axes[0].set_title('Feature Selection Frequency Across All Methods and Folds', fontsize=14)

# Add modality boundaries
start = 0
colors = ['red', 'green', 'blue', 'purple']
for i, (mod, dim) in enumerate(FEATURE_DIMS.items()):
    axes[0].axvline(x=start, color=colors[i], linestyle='--', alpha=0.5, label=f'{mod} start')
    start += dim
axes[0].legend(fontsize=8)
axes[0].grid(axis='y', alpha=0.3)

# Top 20 most and least selected features
top_20_selected = np.argsort(feature_freq_normalized)[::-1][:20]
top_20_removed = np.argsort(feature_freq_normalized)[:20]

x_pos = np.arange(20)
width = 0.35
axes[1].bar(x_pos - width/2, feature_freq_normalized[top_20_selected], width, label='Top 20 Most Selected', color='green', alpha=0.7)
axes[1].bar(x_pos + width/2, feature_freq_normalized[top_20_removed], width, label='Top 20 Least Selected', color='red', alpha=0.7)
axes[1].set_xlabel('Rank')
axes[1].set_ylabel('Selection Frequency')
axes[1].set_title('Most vs Least Frequently Selected Features', fontsize=14)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / '10_feature_frequency.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 10_feature_frequency.png")

# Save feature frequency data
freq_df = pd.DataFrame({
    'feature_index': range(TOTAL_FEATURES),
    'overall_selection_freq': feature_freq_normalized,
})
# Add modality labels
start = 0
modality_labels = []
for mod, dim in FEATURE_DIMS.items():
    modality_labels.extend([mod] * dim)
freq_df['modality'] = modality_labels
freq_df.to_csv(PLOTS_DIR / 'feature_selection_frequency.csv', index=False)
print("Saved: feature_selection_frequency.csv")


Saved: 10_feature_frequency.png
Saved: feature_selection_frequency.csv


In [25]:
# ============================================================================
# PLOT 11: Radar/spider chart comparing methods on multiple metrics
# ============================================================================
from matplotlib.patches import FancyBboxPatch
import matplotlib.patches as mpatches

# Select key metrics for radar
radar_metrics = ['Mean Test Acc (%)', 'Mean F1 Macro (%)', 'Mean Feature Retention (%)', 'Mean Total Time (s)']

# Normalize to [0, 1] for radar
radar_data = df_summary[['Method'] + radar_metrics].copy()
for col in radar_metrics:
    min_val = radar_data[col].min()
    max_val = radar_data[col].max()
    if max_val > min_val:
        radar_data[col] = (radar_data[col] - min_val) / (max_val - min_val)
    else:
        radar_data[col] = 0.5

# For time, invert (lower is better)
radar_data['Mean Total Time (s)'] = 1 - radar_data['Mean Total Time (s)']
radar_data = radar_data.rename(columns={'Mean Total Time (s)': 'Speed (inv. time)'})
radar_cols = ['Mean Test Acc (%)', 'Mean F1 Macro (%)', 'Mean Feature Retention (%)', 'Speed (inv. time)']

# Select subset for readability (standard + top 5 metaheuristics)
top_meta = df_summary[df_summary['Method'].str.startswith('meta_')].nlargest(5, 'Mean Test Acc (%)')['Method'].tolist()
methods_to_plot = ['baseline', 'mutual_info', 'rfe', 'lasso'] + top_meta

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
angles = np.linspace(0, 2 * np.pi, len(radar_cols), endpoint=False).tolist()
angles += angles[:1]

for method in methods_to_plot:
    row = radar_data[radar_data['Method'] == method]
    if len(row) == 0:
        continue
    values = row[radar_cols].values.flatten().tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=1.5, label=method)
    ax.fill(angles, values, alpha=0.05)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(radar_cols, fontsize=9)
ax.set_title('Multi-Metric Comparison (Normalized)', fontsize=14, pad=20)
ax.legend(bbox_to_anchor=(1.3, 1.1), fontsize=8)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '11_radar_comparison.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 11_radar_comparison.png")


Saved: 11_radar_comparison.png


In [26]:
# ============================================================================
# PLOT 12: Standard vs Metaheuristics grouped comparison
# ============================================================================
standard_methods = ['baseline', 'mutual_info', 'rfe', 'lasso']
meta_methods = [m for m in ALL_METHODS if m.startswith('meta_')]

fig, axes = plt.subplots(1, 3, figsize=(20, 7))

# Group averages
std_accs = df_all[df_all['Method'].isin(standard_methods)].groupby('Method')['Test Accuracy (%)'].mean()
meta_accs = df_all[df_all['Method'].isin(meta_methods)].groupby('Method')['Test Accuracy (%)'].mean()

# Accuracy: standard vs meta
all_std = df_all[df_all['Method'].isin(standard_methods)]['Test Accuracy (%)']
all_meta = df_all[df_all['Method'].isin(meta_methods)]['Test Accuracy (%)']
bp = axes[0].boxplot([all_std, all_meta], labels=['Standard\n(MI, RFE, LASSO)', 'Metaheuristics\n(14 optimizers)'],
                      patch_artist=True)
bp['boxes'][0].set_facecolor('lightblue')
bp['boxes'][1].set_facecolor('lightsalmon')
axes[0].set_ylabel('Test Accuracy (%)')
axes[0].set_title('Standard vs Metaheuristic Accuracy', fontsize=13)
axes[0].grid(axis='y', alpha=0.3)

# Feature retention
all_std_f = df_all[(df_all['Method'].isin(standard_methods)) & (df_all['Method'] != 'baseline')]['Feature Retention (%)']
all_meta_f = df_all[df_all['Method'].isin(meta_methods)]['Feature Retention (%)']
bp2 = axes[1].boxplot([all_std_f, all_meta_f], labels=['Standard', 'Metaheuristics'],
                       patch_artist=True)
bp2['boxes'][0].set_facecolor('lightblue')
bp2['boxes'][1].set_facecolor('lightsalmon')
axes[1].set_ylabel('Feature Retention (%)')
axes[1].set_title('Standard vs Metaheuristic Feature Retention', fontsize=13)
axes[1].grid(axis='y', alpha=0.3)

# Time
all_std_t = df_all[(df_all['Method'].isin(standard_methods)) & (df_all['Method'] != 'baseline')]['Total Time (s)']
all_meta_t = df_all[df_all['Method'].isin(meta_methods)]['Total Time (s)']
bp3 = axes[2].boxplot([all_std_t, all_meta_t], labels=['Standard', 'Metaheuristics'],
                       patch_artist=True)
bp3['boxes'][0].set_facecolor('lightblue')
bp3['boxes'][1].set_facecolor('lightsalmon')
axes[2].set_ylabel('Total Time (s)')
axes[2].set_title('Standard vs Metaheuristic Time', fontsize=13)
axes[2].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / '12_standard_vs_metaheuristics.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 12_standard_vs_metaheuristics.png")


Saved: 12_standard_vs_metaheuristics.png


In [27]:
# ============================================================================
# PLOT 13: GPU memory usage comparison
# ============================================================================
fig, ax = plt.subplots(figsize=(16, 7))
gpu_summary = df_all.groupby('Method')['GPU Peak (MB)'].agg(['mean', 'std']).reindex(ALL_METHODS)
bars = ax.bar(gpu_summary.index, gpu_summary['mean'], yerr=gpu_summary['std'], capsize=3, alpha=0.7, color='gold')
ax.set_ylabel('GPU Peak Memory (MB)', fontsize=12)
ax.set_title('GPU Peak Memory Usage by Method', fontsize=16)
plt.xticks(rotation=60, ha='right')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(PLOTS_DIR / '13_gpu_memory.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 13_gpu_memory.png")


Saved: 13_gpu_memory.png


In [28]:
# ============================================================================
# PLOT 14: Per-fold accuracy line plot (all methods)
# ============================================================================
fig, ax = plt.subplots(figsize=(16, 8))

for method in ALL_METHODS:
    mdf = df_all[df_all['Method'] == method].sort_values('Fold')
    if len(mdf) > 0:
        linewidth = 2.5 if method in ['baseline', 'mutual_info', 'rfe', 'lasso'] else 1.0
        alpha = 1.0 if method in ['baseline', 'mutual_info', 'rfe', 'lasso'] else 0.6
        ax.plot(mdf['Fold'], mdf['Test Accuracy (%)'], marker='o', markersize=4,
                label=method, linewidth=linewidth, alpha=alpha)

ax.set_xlabel('Fold', fontsize=12)
ax.set_ylabel('Test Accuracy (%)', fontsize=12)
ax.set_title('Test Accuracy Across Folds (All Methods)', fontsize=16)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
ax.grid(alpha=0.3)
ax.set_xticks(range(1, N_FOLDS + 1))
plt.tight_layout()
plt.savefig(PLOTS_DIR / '14_per_fold_accuracy_lines.png', dpi=300, bbox_inches='tight')
plt.close()
print("Saved: 14_per_fold_accuracy_lines.png")


Saved: 14_per_fold_accuracy_lines.png


## Statistical Significance Tests

In [29]:
from scipy.stats import ttest_rel, wilcoxon

print("\n" + "="*80)
print("STATISTICAL SIGNIFICANCE TESTS (each method vs baseline)")
print("="*80)

baseline_accs = []
for fold_idx in range(N_FOLDS):
    if fold_idx in master_results['baseline']:
        baseline_accs.append(master_results['baseline'][fold_idx]['test_acc'] * 100)

stat_rows = []

for method in ALL_METHODS:
    if method == 'baseline':
        continue
    
    method_accs = []
    for fold_idx in range(N_FOLDS):
        if fold_idx in master_results[method]:
            method_accs.append(master_results[method][fold_idx]['test_acc'] * 100)
    
    if len(method_accs) != len(baseline_accs):
        continue
    
    try:
        t_stat, p_value_t = ttest_rel(method_accs, baseline_accs)
    except:
        t_stat, p_value_t = 0, 1
    
    try:
        w_stat, p_value_w = wilcoxon(method_accs, baseline_accs)
    except:
        w_stat, p_value_w = 0, 1
    
    direction = "better" if np.mean(method_accs) > np.mean(baseline_accs) else "worse"
    significant = "Yes" if p_value_t < 0.05 else "No"
    
    stat_rows.append({
        'Method': method,
        'Mean Acc (%)': np.mean(method_accs),
        'Baseline Acc (%)': np.mean(baseline_accs),
        'Diff (%)': np.mean(method_accs) - np.mean(baseline_accs),
        't-stat': t_stat,
        'p-value (t-test)': p_value_t,
        'W-stat': w_stat,
        'p-value (Wilcoxon)': p_value_w,
        'Direction': direction,
        'Significant (p<0.05)': significant,
    })
    
    print(f"\n{method.upper()} vs BASELINE:")
    print(f"  Method: {np.mean(method_accs):.2f}% vs Baseline: {np.mean(baseline_accs):.2f}%")
    print(f"  Paired t-test: t={t_stat:.4f}, p={p_value_t:.4f}")
    print(f"  Wilcoxon test: W={w_stat:.4f}, p={p_value_w:.4f}")
    print(f"  -> {direction}, {'Significant' if significant == 'Yes' else 'Not significant'}")

stat_df = pd.DataFrame(stat_rows).round(4)
stat_df.to_csv(RESULTS_ROOT / "statistical_tests.csv", index=False)
print(f"\nSaved: statistical_tests.csv")
print("\n" + "="*80)



STATISTICAL SIGNIFICANCE TESTS (each method vs baseline)

MUTUAL_INFO vs BASELINE:
  Method: 93.97% vs Baseline: 93.39%
  Paired t-test: t=0.5518, p=0.5982
  Wilcoxon test: W=10.0000, p=0.5156
  -> better, Not significant

RFE vs BASELINE:
  Method: 94.55% vs Baseline: 93.39%
  Paired t-test: t=1.1433, p=0.2905
  Wilcoxon test: W=6.0000, p=0.4062
  -> better, Not significant

LASSO vs BASELINE:
  Method: 87.01% vs Baseline: 93.39%
  Paired t-test: t=-2.3268, p=0.0529
  Wilcoxon test: W=6.0000, p=0.1094
  -> worse, Not significant

META_BAT vs BASELINE:
  Method: 93.40% vs Baseline: 93.39%
  Paired t-test: t=0.0055, p=0.9958
  Wilcoxon test: W=13.0000, p=0.9375
  -> better, Not significant

META_CS vs BASELINE:
  Method: 93.27% vs Baseline: 93.39%
  Paired t-test: t=-0.0989, p=0.9240
  Wilcoxon test: W=9.0000, p=0.4688
  -> worse, Not significant

META_DE vs BASELINE:
  Method: 93.16% vs Baseline: 93.39%
  Paired t-test: t=-0.1670, p=0.8721
  Wilcoxon test: W=12.0000, p=0.8125
  -> wor

## Final Summary

In [30]:
print("\n" + "="*80)
print("EXPERIMENT COMPLETE - RESULTS SUMMARY")
print("="*80)

print(f"\nOutput directory: {RESULTS_ROOT}")
print(f"\nPer-method folders:")
for method in ALL_METHODS:
    method_dir = RESULTS_ROOT / method
    n_files = len(list(method_dir.glob('*')))
    print(f"  {method_dir}/ ({n_files} files)")

print(f"\nPlots directory: {PLOTS_DIR}")
for p in sorted(PLOTS_DIR.glob('*.png')):
    print(f"  {p.name}")

print(f"\nCSV files:")
for p in sorted(RESULTS_ROOT.glob('*.csv')):
    print(f"  {p.name}")

print(f"\n{'='*80}")
print("KEY FILES:")
print(f"  - all_results_per_fold.csv : Every metric for every fold x method")
print(f"  - summary_all_methods.csv  : Aggregated summary")
print(f"  - statistical_tests.csv    : Significance tests vs baseline")
print(f"  - Per-method folders       : Individual fold JSONs + feature masks")
print(f"  - plots/                   : 14 comprehensive visualizations")
print(f"{'='*80}")

# Print top 5 methods by mean accuracy
print("\nTOP 5 METHODS BY MEAN TEST ACCURACY:")
top5 = df_summary.nlargest(5, 'Mean Test Acc (%)')
for i, row in top5.iterrows():
    print(f"  {row['Method']:20s}: {row['Mean Test Acc (%)']:.2f}% ± {row['Std Test Acc (%)']:.2f}% "
          f"(Features: {row['Mean Feature Retention (%)']:.1f}%)")



EXPERIMENT COMPLETE - RESULTS SUMMARY

Output directory: results_per_fold_new

Per-method folders:
  results_per_fold_new/baseline/ (16 files)
  results_per_fold_new/mutual_info/ (16 files)
  results_per_fold_new/rfe/ (16 files)
  results_per_fold_new/lasso/ (16 files)
  results_per_fold_new/meta_BAT/ (16 files)
  results_per_fold_new/meta_CS/ (16 files)
  results_per_fold_new/meta_DE/ (16 files)
  results_per_fold_new/meta_FFA/ (16 files)
  results_per_fold_new/meta_GA/ (16 files)
  results_per_fold_new/meta_GWO/ (16 files)
  results_per_fold_new/meta_HHO/ (16 files)
  results_per_fold_new/meta_JAYA/ (16 files)
  results_per_fold_new/meta_MFO/ (16 files)
  results_per_fold_new/meta_MVO/ (16 files)
  results_per_fold_new/meta_PSO/ (16 files)
  results_per_fold_new/meta_SCA/ (16 files)
  results_per_fold_new/meta_SSA/ (16 files)
  results_per_fold_new/meta_WOA/ (16 files)

Plots directory: results_per_fold_new/plots
  01_accuracy_heatmap.png
  02_accuracy_boxplot.png
  03_feature_reten

In [31]:
# ============================================================================
# BEST METAHEURISTIC PER FOLD: Test Accuracy & Feature Retention
# ============================================================================

# Identify metaheuristic-only methods
meta_methods = [m for m in ALL_METHODS if m.startswith('meta_')]
modality_names = list(FEATURE_DIMS.keys())  # ['depth', 'sensor', 'skeleton', 'video']

print("=" * 90)
print("BEST METAHEURISTIC PER FOLD (selected by highest test accuracy)")
print("=" * 90)

best_per_fold = []  # collect per-fold best results

for fold_idx in range(N_FOLDS):
    best_method = None
    best_test_acc = -1.0
    best_result = None

    for method_name in meta_methods:
        if fold_idx in master_results[method_name]:
            r = master_results[method_name][fold_idx]
            if r['test_acc'] > best_test_acc:
                best_test_acc = r['test_acc']
                best_method = method_name
                best_result = r

    if best_result is None:
        print(f"  Fold {fold_idx + 1}: No metaheuristic results found!")
        continue

    # Extract modality-wise retention
    mod_ret = best_result['modality_retention']
    mod_retention_pct = {}
    for mod in modality_names:
        if mod in mod_ret and isinstance(mod_ret[mod], dict):
            mod_retention_pct[mod] = mod_ret[mod].get('percentage', 0.0)
        else:
            mod_retention_pct[mod] = float(mod_ret.get(mod, 0.0))

    fold_entry = {
        'Fold': fold_idx + 1,
        'Best Metaheuristic': best_method,
        'Test Accuracy (%)': best_result['test_acc'] * 100,
        'Feature Retention (%)': best_result['feature_retention_pct'],
        'Features Selected': best_result['num_features_selected'],
        'Features Total': best_result['num_features_total'],
    }
    for mod in modality_names:
        fold_entry[f'{mod} Retention (%)'] = mod_retention_pct[mod]

    best_per_fold.append(fold_entry)

    # Print per-fold info
    print(f"\n  Fold {fold_idx + 1}: Best = {best_method}")
    print(f"    Test Accuracy:      {fold_entry['Test Accuracy (%)']:.2f}%")
    print(f"    Feature Retention:  {fold_entry['Feature Retention (%)']:.2f}% "
          f"({fold_entry['Features Selected']}/{fold_entry['Features Total']})")
    mod_str = ', '.join([f"{mod}={mod_retention_pct[mod]:.1f}%" for mod in modality_names])
    print(f"    Modality Retention: {mod_str}")

# Build DataFrame
df_best_meta = pd.DataFrame(best_per_fold)

# ============================================================================
# AVERAGE ACROSS FOLDS
# ============================================================================
print("\n" + "=" * 90)
print("AVERAGE ACROSS ALL FOLDS (Best Metaheuristic per Fold)")
print("=" * 90)

avg_test_acc = df_best_meta['Test Accuracy (%)'].mean()
std_test_acc = df_best_meta['Test Accuracy (%)'].std()
avg_feat_ret = df_best_meta['Feature Retention (%)'].mean()
std_feat_ret = df_best_meta['Feature Retention (%)'].std()

print(f"\n  Average Test Accuracy:     {avg_test_acc:.2f}% ± {std_test_acc:.2f}%")
print(f"  Average Feature Retention: {avg_feat_ret:.2f}% ± {std_feat_ret:.2f}%")

print(f"\n  Average Modality-wise Feature Retention:")
for mod in modality_names:
    col = f'{mod} Retention (%)'
    avg_mod = df_best_meta[col].mean()
    std_mod = df_best_meta[col].std()
    print(f"    {mod:>10s}: {avg_mod:.2f}% ± {std_mod:.2f}%")

# ============================================================================
# SUMMARY TABLE
# ============================================================================
print("\n" + "=" * 90)
print("PER-FOLD SUMMARY TABLE")
print("=" * 90)
display_cols = ['Fold', 'Best Metaheuristic', 'Test Accuracy (%)', 'Feature Retention (%)']
display_cols += [f'{mod} Retention (%)' for mod in modality_names]
print(df_best_meta[display_cols].to_string(index=False, float_format='%.2f'))

# Save to CSV
csv_path = RESULTS_ROOT / "best_metaheuristic_per_fold.csv"
df_best_meta.to_csv(csv_path, index=False)
print(f"\nSaved to: {csv_path}")

BEST METAHEURISTIC PER FOLD (selected by highest test accuracy)

  Fold 1: Best = meta_GA
    Test Accuracy:      99.07%
    Feature Retention:  47.80% (1880/3933)
    Modality Retention: depth=44.0%, sensor=50.3%, skeleton=47.4%, video=47.7%

  Fold 2: Best = meta_JAYA
    Test Accuracy:      94.44%
    Feature Retention:  47.80% (1880/3933)
    Modality Retention: depth=42.6%, sensor=47.4%, skeleton=47.8%, video=48.7%

  Fold 3: Best = meta_BAT
    Test Accuracy:      91.67%
    Feature Retention:  48.94% (1925/3933)
    Modality Retention: depth=54.6%, sensor=50.2%, skeleton=49.4%, video=47.0%

  Fold 4: Best = meta_GA
    Test Accuracy:      92.59%
    Feature Retention:  47.57% (1871/3933)
    Modality Retention: depth=45.8%, sensor=45.7%, skeleton=48.5%, video=47.6%

  Fold 5: Best = meta_CS
    Test Accuracy:      95.37%
    Feature Retention:  47.34% (1862/3933)
    Modality Retention: depth=47.2%, sensor=48.8%, skeleton=47.4%, video=46.6%

  Fold 6: Best = meta_SSA
    Test Ac

In [1]:
"""
UTD-MHAD Feature Selection Results — Comprehensive Summary
Usage: python summarize_results.py
Reads: results_per_fold_new/all_results_per_fold.csv
"""

import pandas as pd
import numpy as np

pd.set_option('display.max_rows', None)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

# Load data
df = pd.read_csv('results_per_fold_new/all_results_per_fold.csv')

# Column names
acc_col = 'Test Accuracy (%)'
ret_col = 'Feature Retention (%)'
params_col = 'Model Params'
size_col = 'Model Size (MB)'
train_col = 'Train Time (s)'
gpu_col = 'GPU Peak (MB)'

methods = df['Method'].unique()
meta_methods = sorted([m for m in methods if m.startswith('meta_')])
std_methods = ['mutual_info', 'rfe', 'lasso']
folds = sorted(df['Fold'].unique())
n_folds = len(folds)

# ============================================================================
# BUILD COMBINED META
# ============================================================================
meta_df = df[df['Method'].isin(meta_methods)]
combined_meta_rows = []

for fold in folds:
    fold_meta = meta_df[meta_df['Fold'] == fold]
    best_idx = fold_meta[acc_col].idxmax()
    best_row = fold_meta.loc[best_idx].copy()
    row_dict = best_row.to_dict()
    row_dict['_winner'] = row_dict['Method']
    row_dict['Method'] = 'combined_meta'
    combined_meta_rows.append(row_dict)

df_cm = pd.DataFrame(combined_meta_rows)

# Comparison df: baseline + standard + combined_meta
df_compare = pd.concat([
    df[df['Method'].isin(['baseline'] + std_methods)],
    df_cm.drop(columns=['_winner'])
], ignore_index=True)

compare_order = ['baseline', 'mutual_info', 'rfe', 'lasso', 'combined_meta']

# Baseline reference values
bl = df[df['Method'] == 'baseline']
bl_acc = bl[acc_col].mean()
bl_params = bl[params_col].mean()
bl_size = bl[size_col].mean()
bl_train = bl[train_col].mean()
bl_gpu = bl[gpu_col].mean()


def header(title, section_num):
    print(f"\n\n{'='*110}")
    print(f"{section_num}. {title}")
    print(f"{'='*110}")


# ============================================================================
# 1. MAIN 5-METHOD COMPARISON
# ============================================================================
header("MAIN COMPARISON (baseline, MI, RFE, LASSO, combined_meta)", 1)

print(f"\n  {'Method':<20s} {'Acc (%)':>14s}  {'Ret (%)':>10s}  "
      f"{'Params':>18s}  {'Size (MB)':>14s}  {'Train (s)':>14s}  {'GPU (MB)':>12s}")
print(f"  {'-'*20} {'-'*14}  {'-'*10}  {'-'*18}  {'-'*14}  {'-'*14}  {'-'*12}")

for method in compare_order:
    m = df_compare[df_compare['Method'] == method]
    print(f"  {method:<20s} "
          f"{m[acc_col].mean():6.2f} ± {m[acc_col].std():5.2f}  "
          f"{m[ret_col].mean():6.1f}     "
          f"{m[params_col].mean():>9,.0f} ± {m[params_col].std():>6,.0f}  "
          f"{m[size_col].mean():6.3f} ± {m[size_col].std():4.3f}  "
          f"{m[train_col].mean():6.2f} ± {m[train_col].std():4.2f}  "
          f"{m[gpu_col].mean():6.2f}")


# ============================================================================
# 2. INDIVIDUAL METAHEURISTIC BREAKDOWN
# ============================================================================
header("INDIVIDUAL METAHEURISTIC BREAKDOWN", 2)

meta_summary = meta_df.groupby('Method').agg(
    Acc_Mean=(acc_col, 'mean'), Acc_Std=(acc_col, 'std'),
    Ret_Mean=(ret_col, 'mean'), Ret_Std=(ret_col, 'std'),
    Params_Mean=(params_col, 'mean'), Params_Std=(params_col, 'std'),
    Size_Mean=(size_col, 'mean'), Size_Std=(size_col, 'std'),
    Train_Mean=(train_col, 'mean'),
    GPU_Mean=(gpu_col, 'mean'),
).round(2).sort_values('Acc_Mean', ascending=False)

# Which meta won each fold
meta_winners = df_cm['_winner'].value_counts()

print(f"\n  {'Method':<15s} {'Acc (%)':>14s}  {'Ret (%)':>12s}  "
      f"{'Params':>18s}  {'Size (MB)':>14s}  {'Train (s)':>8s}  {'GPU (MB)':>8s}  Wins")
print(f"  {'-'*15} {'-'*14}  {'-'*12}  {'-'*18}  {'-'*14}  {'-'*8}  {'-'*8}  {'-'*4}")

for method, row in meta_summary.iterrows():
    wins = meta_winners.get(method, 0)
    win_str = f"  ★ {wins}" if wins > 0 else ""
    print(f"  {method:<15s} "
          f"{row.Acc_Mean:6.2f} ± {row.Acc_Std:5.2f}  "
          f"{row.Ret_Mean:5.1f} ± {row.Ret_Std:4.1f}  "
          f"{row.Params_Mean:>9,.0f} ± {row.Params_Std:>6,.0f}  "
          f"{row.Size_Mean:6.3f} ± {row.Size_Std:4.3f}  "
          f"{row.Train_Mean:6.2f}  "
          f"{row.GPU_Mean:6.2f}{win_str}")

print(f"\n  Combined Meta (best/fold):")
print(f"    Acc  = {df_cm[acc_col].mean():.2f}% ± {df_cm[acc_col].std():.2f}%")
print(f"    Ret  = {df_cm[ret_col].mean():.1f}% ± {df_cm[ret_col].std():.1f}%")
print(f"    Per-fold winners: {dict(meta_winners.sort_index())}")


# ============================================================================
# 3. MODEL SIZE & PARAMS ANALYSIS
# ============================================================================
header("MODEL SIZE & PARAMETERS", 3)

print(f"\n  --- By category ---")
print(f"  {'Category':<35s} {'Params':>18s}  {'Size (MB)':>18s}")
print(f"  {'-'*35} {'-'*18}  {'-'*18}")

for label, sub in [
    ('Baseline', bl),
    ('Standard (MI, RFE, LASSO)', df[df['Method'].isin(std_methods)]),
    ('Metaheuristics (all 14)', meta_df),
    ('Combined Meta (best/fold)', df_cm),
]:
    print(f"  {label:<35s} "
          f"{sub[params_col].mean():>9,.0f} ± {sub[params_col].std():>6,.0f}  "
          f"{sub[size_col].mean():>7.3f} ± {sub[size_col].std():>5.3f}")

# Param reduction vs baseline
print(f"\n  --- Parameter reduction vs baseline ({bl_params:,.0f} params / {bl_size:.3f} MB) ---")
for method in std_methods + ['combined_meta']:
    m = df_compare[df_compare['Method'] == method]
    p = m[params_col].mean()
    s = m[size_col].mean()
    print(f"  {method:<20s}: {p:>9,.0f} params ({(1-p/bl_params)*100:>5.1f}% reduction)  "
          f"{s:.3f} MB ({(1-s/bl_size)*100:>5.1f}% reduction)")

print(f"\n  --- Individual metaheuristic model sizes (sorted smallest → largest) ---")
meta_size_sorted = meta_summary.sort_values('Params_Mean')
for method, row in meta_size_sorted.iterrows():
    name = method.replace('meta_', '')
    reduction = (1 - row.Params_Mean / bl_params) * 100
    bar = "█" * max(1, int(row.Params_Mean / 10000))
    print(f"  {name:>5s}: {row.Params_Mean:>9,.0f} params  {row.Size_Mean:.3f} MB  "
          f"({reduction:>5.1f}% smaller)  {bar}")

# Grouping by aggressiveness
print(f"\n  --- Aggressiveness tiers ---")
aggressive = meta_size_sorted[meta_size_sorted['Params_Mean'] < 100000]
moderate = meta_size_sorted[(meta_size_sorted['Params_Mean'] >= 100000) & (meta_size_sorted['Params_Mean'] < 250000)]
conservative = meta_size_sorted[meta_size_sorted['Params_Mean'] >= 250000]

if len(aggressive) > 0:
    names = [m.replace('meta_', '') for m in aggressive.index]
    print(f"  Aggressive  (<100K params):  {', '.join(names)}")
    print(f"    Avg: {aggressive.Params_Mean.mean():,.0f} params, "
          f"{aggressive.Size_Mean.mean():.3f} MB, "
          f"Acc = {aggressive.Acc_Mean.mean():.2f}%")

if len(moderate) > 0:
    names = [m.replace('meta_', '') for m in moderate.index]
    print(f"  Moderate  (100K–250K params): {', '.join(names)}")
    print(f"    Avg: {moderate.Params_Mean.mean():,.0f} params, "
          f"{moderate.Size_Mean.mean():.3f} MB, "
          f"Acc = {moderate.Acc_Mean.mean():.2f}%")

if len(conservative) > 0:
    names = [m.replace('meta_', '') for m in conservative.index]
    print(f"  Conservative (>250K params): {', '.join(names)}")
    print(f"    Avg: {conservative.Params_Mean.mean():,.0f} params, "
          f"{conservative.Size_Mean.mean():.3f} MB, "
          f"Acc = {conservative.Acc_Mean.mean():.2f}%")


# ============================================================================
# 4. TRAIN TIME ANALYSIS
# ============================================================================
header("TRAIN TIME", 4)

print(f"\n  --- By category ---")
print(f"  {'Category':<35s} {'Train Time (s)':>18s}  {'Range':>16s}")
print(f"  {'-'*35} {'-'*18}  {'-'*16}")

for label, sub in [
    ('Baseline', bl),
    ('Standard (MI, RFE, LASSO)', df[df['Method'].isin(std_methods)]),
    ('Metaheuristics (all 14)', meta_df),
    ('Combined Meta (best/fold)', df_cm),
]:
    print(f"  {label:<35s} "
          f"{sub[train_col].mean():>7.2f} ± {sub[train_col].std():>5.2f}  "
          f"[{sub[train_col].min():.2f}–{sub[train_col].max():.2f}]")

print(f"\n  --- Per method (sorted) ---")
train_by_method = df.groupby('Method')[train_col].agg(['mean', 'std', 'min', 'max']).round(2)
train_by_method = train_by_method.sort_values('mean')
for method, row in train_by_method.iterrows():
    print(f"  {method:<20s}: {row['mean']:6.2f} ± {row['std']:4.2f}s  "
          f"[{row['min']:.2f}–{row['max']:.2f}]")

overall_min = df[train_col].min()
overall_max = df[train_col].max()
overall_range = overall_max - overall_min
print(f"\n  Overall range: {overall_min:.2f}–{overall_max:.2f}s (spread: {overall_range:.2f}s)")
print(f"  Takeaway: Train time is {'very consistent' if overall_range < 5 else 'fairly consistent' if overall_range < 10 else 'variable'} "
      f"across methods ({overall_range:.1f}s spread).")


# ============================================================================
# 5. GPU PEAK MEMORY ANALYSIS
# ============================================================================
header("GPU PEAK MEMORY (MB)", 5)

print(f"\n  --- By category ---")
print(f"  {'Category':<35s} {'GPU Peak (MB)':>18s}  {'Range':>16s}")
print(f"  {'-'*35} {'-'*18}  {'-'*16}")

for label, sub in [
    ('Baseline', bl),
    ('Standard (MI, RFE, LASSO)', df[df['Method'].isin(std_methods)]),
    ('Metaheuristics (all 14)', meta_df),
    ('Combined Meta (best/fold)', df_cm),
]:
    print(f"  {label:<35s} "
          f"{sub[gpu_col].mean():>7.2f} ± {sub[gpu_col].std():>5.2f}  "
          f"[{sub[gpu_col].min():.2f}–{sub[gpu_col].max():.2f}]")

print(f"\n  --- Per method (sorted by GPU usage) ---")
gpu_by_method = df.groupby('Method')[gpu_col].agg(['mean', 'std', 'min', 'max']).round(2)
gpu_by_method = gpu_by_method.sort_values('mean')
for method, row in gpu_by_method.iterrows():
    bar = "█" * max(1, int(row['mean'] / 2))
    print(f"  {method:<20s}: {row['mean']:6.2f} ± {row['std']:4.2f} MB  "
          f"[{row['min']:.2f}–{row['max']:.2f}]  {bar}")

print(f"\n  --- GPU memory tiers ---")
low_gpu = gpu_by_method[gpu_by_method['mean'] < 20]
mid_gpu = gpu_by_method[(gpu_by_method['mean'] >= 20) & (gpu_by_method['mean'] < 24)]
high_gpu = gpu_by_method[gpu_by_method['mean'] >= 24]

if len(low_gpu) > 0:
    names = [m.replace('meta_', '') if m.startswith('meta_') else m for m in low_gpu.index]
    print(f"  Low   (<20 MB):  {', '.join(names)}  — avg {low_gpu['mean'].mean():.2f} MB")
if len(mid_gpu) > 0:
    names = [m.replace('meta_', '') if m.startswith('meta_') else m for m in mid_gpu.index]
    print(f"  Mid   (20–24 MB): {', '.join(names)}  — avg {mid_gpu['mean'].mean():.2f} MB")
if len(high_gpu) > 0:
    names = [m.replace('meta_', '') if m.startswith('meta_') else m for m in high_gpu.index]
    print(f"  High  (>24 MB):  {', '.join(names)}  — avg {high_gpu['mean'].mean():.2f} MB")

overall_gpu_range = df[gpu_col].max() - df[gpu_col].min()
print(f"\n  Overall range: {df[gpu_col].min():.2f}–{df[gpu_col].max():.2f} MB "
      f"(spread: {overall_gpu_range:.2f} MB)")
print(f"  Takeaway: All models are tiny. The {overall_gpu_range:.0f} MB spread is negligible "
      f"for any modern GPU.")


# ============================================================================
# 6. ACCURACY vs EFFICIENCY TRADEOFF
# ============================================================================
header("ACCURACY vs EFFICIENCY TRADEOFF", 6)

# Per individual meta
print(f"\n  --- Individual metaheuristics: Accuracy vs Model Size ---")
print(f"  {'Method':<15s} {'Acc (%)':>10s}  {'Params':>10s}  {'Size (MB)':>10s}  "
      f"{'Acc/10K params':>14s}  Assessment")
print(f"  {'-'*15} {'-'*10}  {'-'*10}  {'-'*10}  {'-'*14}  {'-'*25}")

for method, row in meta_summary.sort_values('Acc_Mean', ascending=False).iterrows():
    name = method.replace('meta_', '')
    efficiency = row.Acc_Mean / (row.Params_Mean / 10000)
    
    # Classify
    if row.Acc_Mean > bl_acc and row.Params_Mean < bl_params * 0.7:
        assessment = "★ Sweet spot"
    elif row.Acc_Mean > bl_acc:
        assessment = "✓ Beats baseline"
    elif row.Params_Mean < bl_params * 0.3:
        assessment = "⚡ Very compact, lower acc"
    elif row.Params_Mean < bl_params * 0.6:
        assessment = "↓ Compact, lower acc"
    else:
        assessment = "~ Similar size, lower acc"
    
    print(f"  {name:<15s} {row.Acc_Mean:>8.2f}%  {row.Params_Mean:>10,.0f}  "
          f"{row.Size_Mean:>8.3f}  {efficiency:>12.2f}  {assessment}")

# The 5 comparison methods
print(f"\n  --- 5-method comparison: Accuracy vs Model Size ---")
print(f"  {'Method':<20s} {'Acc (%)':>10s}  {'Params':>10s}  {'Size (MB)':>10s}  "
      f"{'vs Baseline':>12s}")
print(f"  {'-'*20} {'-'*10}  {'-'*10}  {'-'*10}  {'-'*12}")

for method in compare_order:
    m = df_compare[df_compare['Method'] == method]
    acc = m[acc_col].mean()
    params = m[params_col].mean()
    size = m[size_col].mean()
    diff = acc - bl_acc
    print(f"  {method:<20s} {acc:>8.2f}%  {params:>10,.0f}  {size:>8.3f}  "
          f"{diff:>+10.2f}%")


# ============================================================================
# 7. BEST METHOD PER FOLD (5-method comparison)
# ============================================================================
header("BEST METHOD PER FOLD (5-method comparison)", 7)

win_counts = {m: 0 for m in compare_order}
second_counts = {m: 0 for m in compare_order}
third_counts = {m: 0 for m in compare_order}

for fold in folds:
    fold_df = df_compare[df_compare['Fold'] == fold]
    fold_ranked = fold_df.sort_values(acc_col, ascending=False)
    ranked = list(zip(fold_ranked['Method'], fold_ranked[acc_col]))
    
    win_counts[ranked[0][0]] += 1
    if len(ranked) > 1: second_counts[ranked[1][0]] += 1
    if len(ranked) > 2: third_counts[ranked[2][0]] += 1
    
    top3 = [f"{m:15s} ({a:.2f}%)" for m, a in ranked[:3]]
    print(f"  Fold {fold}: 🥇 {top3[0]}  🥈 {top3[1]}  🥉 {top3[2]}")

print(f"\n  Podium summary:")
print(f"  {'Method':<20s} {'🥇':>4s}  {'🥈':>4s}  {'🥉':>4s}  Total top-3")
print(f"  {'-'*20} {'-'*4}  {'-'*4}  {'-'*4}  {'-'*11}")
for method in compare_order:
    total = win_counts[method] + second_counts[method] + third_counts[method]
    print(f"  {method:<20s} {win_counts[method]:>4d}  {second_counts[method]:>4d}  "
          f"{third_counts[method]:>4d}  {total:>5d}")


# ============================================================================
# 8. FOLD DIFFICULTY
# ============================================================================
header("FOLD DIFFICULTY (avg accuracy across all methods)", 8)

fold_diff = df.groupby('Fold')[acc_col].agg(['mean', 'std', 'min', 'max']).round(2)
fold_diff = fold_diff.sort_values('mean')

for fold, row in fold_diff.iterrows():
    bar = "█" * int(row['mean'] / 2)
    print(f"  Fold {fold}: {row['mean']:6.2f}% ± {row['std']:5.2f}%  "
          f"[{row['min']:.1f}–{row['max']:.1f}]  {bar}")

print(f"\n  Easiest: Fold {fold_diff['mean'].idxmax()} ({fold_diff['mean'].max():.2f}%)")
print(f"  Hardest: Fold {fold_diff['mean'].idxmin()} ({fold_diff['mean'].min():.2f}%)")


# ============================================================================
# 9. STABILITY
# ============================================================================
header("STABILITY (consistency across folds)", 9)

print(f"\n  --- 5-method comparison ---")
for method in compare_order:
    m = df_compare[df_compare['Method'] == method]
    rng = m[acc_col].max() - m[acc_col].min()
    print(f"  {method:<20s}: {m[acc_col].mean():.2f}% ± {m[acc_col].std():.2f}%  "
          f"(range: {rng:.1f} pp)")

print(f"\n  --- Individual metaheuristics (sorted by std, most stable first) ---")
meta_stab = meta_df.groupby('Method')[acc_col].agg(['mean', 'std']).round(2).sort_values('std')
for method, row in meta_stab.iterrows():
    name = method.replace('meta_', '')
    label = "← most stable" if method == meta_stab.index[0] else \
            "← least stable" if method == meta_stab.index[-1] else ""
    print(f"  {name:<10s}: {row['mean']:.2f}% ± {row['std']:.2f}%  {label}")


# ============================================================================
# 10. OVERALL SUMMARY
# ============================================================================
header("OVERALL SUMMARY", 10)

cm_acc = df_cm[acc_col].mean()
cm_params = df_cm[params_col].mean()
cm_size = df_cm[size_col].mean()
cm_ret = df_cm[ret_col].mean()

best_individual = meta_summary.index[0]
best_individual_acc = meta_summary.iloc[0].Acc_Mean

rfe_acc = df[df['Method'] == 'rfe'][acc_col].mean()

print(f"""
  DATASET: UTD-MHAD  |  {n_folds} LOSO folds  |  {len(methods)} methods  |  {len(df)} experiments

  ACCURACY:
    Baseline:                  {bl_acc:.2f}%
    Best standard method:      RFE at {rfe_acc:.2f}% ({rfe_acc - bl_acc:+.2f}% vs baseline)
    Best individual meta:      {best_individual} at {best_individual_acc:.2f}% ({best_individual_acc - bl_acc:+.2f}%)
    Combined Meta (best/fold): {cm_acc:.2f}% ({cm_acc - bl_acc:+.2f}%)

  MODEL EFFICIENCY:
    Baseline:       {bl_params:>9,.0f} params  /  {bl_size:.3f} MB
    Standard (50%): {df[df['Method'].isin(std_methods)][params_col].mean():>9,.0f} params  /  {df[df['Method'].isin(std_methods)][size_col].mean():.3f} MB  ({(1 - df[df['Method'].isin(std_methods)][params_col].mean()/bl_params)*100:.0f}% reduction)
    Combined Meta:  {cm_params:>9,.0f} params  /  {cm_size:.3f} MB  ({(1 - cm_params/bl_params)*100:.0f}% reduction)

  COMPUTE:
    Train time range: {df[train_col].min():.1f}–{df[train_col].max():.1f}s (negligible difference)
    GPU peak range:   {df[gpu_col].min():.1f}–{df[gpu_col].max():.1f} MB (all tiny, negligible)

  META SPECTRUM (model size):
    Aggressive (SCA, HHO, WOA, GWO): ~{aggressive.Params_Mean.mean() if len(aggressive) > 0 else 0:,.0f} params, smaller models, accuracy tradeoff
    Conservative (SSA, MVO, CS, FFA, JAYA, etc.): ~{conservative.Params_Mean.mean() if len(conservative) > 0 else 0:,.0f} params, near-standard size, competitive accuracy
    Sweet spot: Methods that beat baseline with meaningful compression

  WINS ({n_folds} folds):
    Combined Meta: 🥇×{win_counts.get('combined_meta', 0)}  🥈×{second_counts.get('combined_meta', 0)}  🥉×{third_counts.get('combined_meta', 0)}
    RFE:           🥇×{win_counts.get('rfe', 0)}  🥈×{second_counts.get('rfe', 0)}  🥉×{third_counts.get('rfe', 0)}
""")

print("=" * 110)
print("DONE")
print("=" * 110)



1. MAIN COMPARISON (baseline, MI, RFE, LASSO, combined_meta)

  Method                      Acc (%)     Ret (%)              Params       Size (MB)       Train (s)      GPU (MB)
  -------------------- --------------  ----------  ------------------  --------------  --------------  ------------
  baseline              93.39 ±  4.68   100.0     2,154,011 ±      0   8.223 ± 0.000    8.09 ± 0.15   67.05
  mutual_info           93.97 ±  2.97    50.0     1,146,907 ±      0   4.381 ± 0.000    6.39 ± 0.49   44.47
  rfe                   94.55 ±  4.73    50.0     1,146,907 ±      0   4.381 ± 0.000    6.41 ± 0.44   44.47
  lasso                 87.01 ±  7.23    50.0     1,146,907 ±      0   4.381 ± 0.000    6.29 ± 0.19   44.47
  combined_meta         96.18 ±  3.10    47.7     1,101,211 ± 16,848   4.207 ± 0.064    6.20 ± 0.03   44.31


2. INDIVIDUAL METAHEURISTIC BREAKDOWN

  Method                 Acc (%)       Ret (%)              Params       Size (MB)  Train (s)  GPU (MB)  Wins
  -----------

In [2]:
"""
UTD-MHAD Feature Selection Results — V1 vs V2 Side-by-Side Summary
Usage: python summarize_results.py

Reads:
  - results_per_fold/all_results_per_fold.csv      (V1)
  - results_per_fold_new/all_results_per_fold.csv   (V2)

Adjust paths below if needed.
"""

import pandas as pd
import numpy as np
import sys

pd.set_option('display.max_rows', None)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', lambda x: f'{x:.2f}')

# ============================================================================
# CONFIGURATION — adjust paths here
# ============================================================================
DATASETS = {
    'V1': 'results_per_fold/all_results_per_fold.csv',
    'V2': 'results_per_fold_new/all_results_per_fold.csv',
}

# Column names
acc_col = 'Test Accuracy (%)'
ret_col = 'Feature Retention (%)'
params_col = 'Model Params'
size_col = 'Model Size (MB)'
train_col = 'Train Time (s)'
gpu_col = 'GPU Peak (MB)'

ALL_COLS = [acc_col, ret_col, params_col, size_col, train_col, gpu_col]


# ============================================================================
# LOAD & PREPARE DATA
# ============================================================================
data = {}  # key: version name, value: dict of useful things

for version, path in DATASETS.items():
    try:
        df = pd.read_csv(path)
    except FileNotFoundError:
        print(f"WARNING: {path} not found, skipping {version}")
        continue

    methods = df['Method'].unique()
    meta_methods = sorted([m for m in methods if m.startswith('meta_')])
    std_methods = ['mutual_info', 'rfe', 'lasso']
    folds = sorted(df['Fold'].unique())
    meta_df = df[df['Method'].isin(meta_methods)]

    # Build combined meta
    cm_rows = []
    for fold in folds:
        fold_meta = meta_df[meta_df['Fold'] == fold]
        best_idx = fold_meta[acc_col].idxmax()
        best_row = fold_meta.loc[best_idx].copy().to_dict()
        best_row['_winner'] = best_row['Method']
        best_row['Method'] = 'combined_meta'
        cm_rows.append(best_row)
    df_cm = pd.DataFrame(cm_rows)

    # Comparison df
    df_compare = pd.concat([
        df[df['Method'].isin(['baseline'] + std_methods)],
        df_cm.drop(columns=['_winner'])
    ], ignore_index=True)

    bl = df[df['Method'] == 'baseline']

    data[version] = {
        'df': df,
        'meta_df': meta_df,
        'df_cm': df_cm,
        'df_compare': df_compare,
        'meta_methods': meta_methods,
        'std_methods': std_methods,
        'folds': folds,
        'meta_winners': df_cm['_winner'].value_counts(),
        'bl': {col: bl[col].mean() for col in ALL_COLS},
    }

versions = list(data.keys())
compare_order = ['baseline', 'mutual_info', 'rfe', 'lasso', 'combined_meta']

W = 55  # column width per version


def header(title, section_num):
    print(f"\n\n{'='*130}")
    print(f"{section_num}. {title}")
    print(f"{'='*130}")


def side_label():
    """Print version labels as column headers"""
    labels = "  " + " " * 22
    for v in versions:
        labels += f"{'│ ' + v:>{W}s}"
    print(labels)
    print("  " + "-" * 22 + ("─" * W) * len(versions))


# ============================================================================
# 1. MAIN 5-METHOD COMPARISON
# ============================================================================
header("MAIN 5-METHOD COMPARISON (mean ± std across folds)", 1)

for metric_name, col, fmt_val, fmt_std in [
    ("Test Accuracy (%)", acc_col, "{:6.2f}", "{:5.2f}"),
    ("Feature Retention (%)", ret_col, "{:6.1f}", "{:5.1f}"),
    ("Model Params", params_col, "{:>9,.0f}", "{:>7,.0f}"),
    ("Model Size (MB)", size_col, "{:7.3f}", "{:5.3f}"),
    ("Train Time (s)", train_col, "{:6.2f}", "{:4.2f}"),
    ("GPU Peak (MB)", gpu_col, "{:6.2f}", "{:4.2f}"),
]:
    print(f"\n  --- {metric_name} ---")
    side_label()
    for method in compare_order:
        line = f"  {method:<22s}"
        for v in versions:
            m = data[v]['df_compare'][data[v]['df_compare']['Method'] == method]
            mean = m[col].mean()
            std = m[col].std()
            cell = f"{fmt_val.format(mean)} ± {fmt_std.format(std)}"
            line += f"{'│ ' + cell:>{W}s}"
        print(line)


# ============================================================================
# 2. INDIVIDUAL METAHEURISTIC BREAKDOWN
# ============================================================================
header("INDIVIDUAL METAHEURISTIC BREAKDOWN", 2)

# Get union of all meta methods
all_metas = sorted(set().union(*[set(data[v]['meta_methods']) for v in versions]))

for metric_name, col, fmt in [
    ("Test Accuracy (%)", acc_col, "{:6.2f}% ± {:5.2f}%"),
    ("Model Params", params_col, "{:>9,.0f} ± {:>6,.0f}"),
    ("Model Size (MB)", size_col, "{:6.3f} ± {:5.3f}"),
    ("GPU Peak (MB)", gpu_col, "{:6.2f} ± {:4.2f}"),
]:
    print(f"\n  --- {metric_name} (sorted by V1 mean) ---")
    
    # Sort by V1 mean descending
    sort_key = {}
    for m in all_metas:
        if versions[0] in data:
            mdf = data[versions[0]]['meta_df']
            sub = mdf[mdf['Method'] == m]
            sort_key[m] = sub[col].mean() if len(sub) > 0 else -999
        else:
            sort_key[m] = -999
    sorted_metas = sorted(all_metas, key=lambda x: -sort_key[x])
    
    side_label()
    for method in sorted_metas:
        name = method.replace('meta_', '')
        line = f"  {name:<22s}"
        for v in versions:
            sub = data[v]['meta_df'][data[v]['meta_df']['Method'] == method]
            if len(sub) > 0:
                mean = sub[col].mean()
                std = sub[col].std()
                wins = data[v]['meta_winners'].get(method, 0)
                cell = fmt.format(mean, std)
                if wins > 0:
                    cell += f" ★{wins}"
                line += f"{'│ ' + cell:>{W}s}"
            else:
                line += f"{'│ —':>{W}s}"
        print(line)

    # Combined meta summary
    line_cm = f"  {'COMBINED META':<22s}"
    for v in versions:
        cm = data[v]['df_cm']
        mean = cm[col].mean()
        std = cm[col].std()
        cell = fmt.format(mean, std)
        line_cm += f"{'│ ' + cell:>{W}s}"
    print(f"  {'─'*22}" + "─" * (W * len(versions)))
    print(line_cm)


# ============================================================================
# 3. MODEL SIZE & PARAMS — DETAILED
# ============================================================================
header("MODEL SIZE & PARAMETERS", 3)

print(f"\n  --- Category averages ---")
side_label()

cat_defs = [
    ('Baseline', lambda v: data[v]['df'][data[v]['df']['Method'] == 'baseline']),
    ('Standard (MI,RFE,LASSO)', lambda v: data[v]['df'][data[v]['df']['Method'].isin(data[v]['std_methods'])]),
    ('Metaheuristics (all 14)', lambda v: data[v]['meta_df']),
    ('Combined Meta', lambda v: data[v]['df_cm']),
]

for label, get_sub in cat_defs:
    line = f"  {label:<22s}"
    for v in versions:
        sub = get_sub(v)
        cell = f"{sub[params_col].mean():>9,.0f} p / {sub[size_col].mean():.3f} MB"
        line += f"{'│ ' + cell:>{W}s}"
    print(line)

# Reduction vs baseline
print(f"\n  --- Parameter reduction vs baseline ---")
side_label()
for method in ['mutual_info', 'rfe', 'lasso', 'combined_meta']:
    line = f"  {method:<22s}"
    for v in versions:
        m = data[v]['df_compare'][data[v]['df_compare']['Method'] == method]
        p = m[params_col].mean()
        bl_p = data[v]['bl'][params_col]
        reduction = (1 - p / bl_p) * 100
        cell = f"{p:>9,.0f} ({reduction:>5.1f}% ↓)"
        line += f"{'│ ' + cell:>{W}s}"
    print(line)

# Aggressiveness tiers per version
print(f"\n  --- Metaheuristic aggressiveness tiers ---")
for v in versions:
    print(f"\n    [{v}]")
    meta_params = data[v]['meta_df'].groupby('Method')[params_col].mean().sort_values()
    bl_p = data[v]['bl'][params_col]
    
    tiers = {'Aggressive (<25% baseline)': [], 'Moderate (25-60%)': [], 'Conservative (>60%)': []}
    for method, p in meta_params.items():
        ratio = p / bl_p
        name = method.replace('meta_', '')
        acc = data[v]['meta_df'][data[v]['meta_df']['Method'] == method][acc_col].mean()
        entry = f"{name}({p:,.0f}p, {acc:.1f}%)"
        if ratio < 0.25:
            tiers['Aggressive (<25% baseline)'].append(entry)
        elif ratio < 0.60:
            tiers['Moderate (25-60%)'].append(entry)
        else:
            tiers['Conservative (>60%)'].append(entry)
    
    for tier_name, entries in tiers.items():
        if entries:
            print(f"      {tier_name}: {', '.join(entries)}")


# ============================================================================
# 4. TRAIN TIME
# ============================================================================
header("TRAIN TIME", 4)

side_label()
for label, get_sub in cat_defs:
    line = f"  {label:<22s}"
    for v in versions:
        sub = get_sub(v)
        cell = f"{sub[train_col].mean():.2f} ± {sub[train_col].std():.2f}s"
        line += f"{'│ ' + cell:>{W}s}"
    print(line)

print(f"\n  --- Takeaway ---")
for v in versions:
    spread = data[v]['df'][train_col].max() - data[v]['df'][train_col].min()
    verdict = "negligible" if spread < 5 else "minor" if spread < 15 else "notable"
    print(f"    {v}: range {data[v]['df'][train_col].min():.1f}–{data[v]['df'][train_col].max():.1f}s "
          f"(spread: {spread:.1f}s) → {verdict} difference")


# ============================================================================
# 5. GPU PEAK MEMORY
# ============================================================================
header("GPU PEAK MEMORY (MB)", 5)

side_label()
for label, get_sub in cat_defs:
    line = f"  {label:<22s}"
    for v in versions:
        sub = get_sub(v)
        cell = f"{sub[gpu_col].mean():.2f} ± {sub[gpu_col].std():.2f} MB"
        line += f"{'│ ' + cell:>{W}s}"
    print(line)

# Per-meta GPU
print(f"\n  --- Per metaheuristic (sorted by V1) ---")
side_label()
gpu_sort = {}
for m in all_metas:
    sub = data[versions[0]]['meta_df'][data[versions[0]]['meta_df']['Method'] == m]
    gpu_sort[m] = sub[gpu_col].mean() if len(sub) > 0 else 999
sorted_metas_gpu = sorted(all_metas, key=lambda x: gpu_sort[x])

for method in sorted_metas_gpu:
    name = method.replace('meta_', '')
    line = f"  {name:<22s}"
    for v in versions:
        sub = data[v]['meta_df'][data[v]['meta_df']['Method'] == method]
        if len(sub) > 0:
            cell = f"{sub[gpu_col].mean():.2f} ± {sub[gpu_col].std():.2f} MB"
        else:
            cell = "—"
        line += f"{'│ ' + cell:>{W}s}"
    print(line)

print(f"\n  --- Takeaway ---")
for v in versions:
    lo = data[v]['df'][gpu_col].min()
    hi = data[v]['df'][gpu_col].max()
    spread = hi - lo
    if spread < 15:
        verdict = "All models tiny, GPU differences negligible"
    elif spread < 30:
        verdict = "Moderate spread — aggressive FS methods noticeably lighter"
    else:
        verdict = "Significant spread — feature selection meaningfully reduces GPU usage"
    print(f"    {v}: {lo:.1f}–{hi:.1f} MB (spread: {spread:.1f} MB) → {verdict}")


# ============================================================================
# 6. ACCURACY vs EFFICIENCY TRADEOFF
# ============================================================================
header("ACCURACY vs EFFICIENCY TRADEOFF", 6)

for v in versions:
    print(f"\n  [{v}] Individual metaheuristics:")
    print(f"  {'Method':<12s} {'Acc(%)':>8s}  {'Params':>10s}  {'Size(MB)':>8s}  "
          f"{'ParamRed':>8s}  Assessment")
    print(f"  {'-'*12} {'-'*8}  {'-'*10}  {'-'*8}  {'-'*8}  {'-'*30}")

    meta_s = data[v]['meta_df'].groupby('Method').agg({
        acc_col: 'mean', params_col: 'mean', size_col: 'mean'
    }).sort_values(acc_col, ascending=False)
    bl_acc = data[v]['bl'][acc_col]
    bl_p = data[v]['bl'][params_col]

    for method, row in meta_s.iterrows():
        name = method.replace('meta_', '')
        reduction = (1 - row[params_col] / bl_p) * 100

        if row[acc_col] > bl_acc and reduction > 30:
            assessment = "★ Sweet spot (beats BL + compact)"
        elif row[acc_col] > bl_acc:
            assessment = "✓ Beats baseline"
        elif reduction > 75:
            assessment = "⚡ Very compact, lower acc"
        elif reduction > 40:
            assessment = "↓ Compact, lower acc"
        else:
            assessment = "~ Similar size, lower acc"

        print(f"  {name:<12s} {row[acc_col]:>7.2f}  {row[params_col]:>10,.0f}  "
              f"{row[size_col]:>7.3f}  {reduction:>7.1f}%  {assessment}")


# ============================================================================
# 7. BEST METHOD PER FOLD
# ============================================================================
header("BEST METHOD PER FOLD (5-method comparison)", 7)

for v in versions:
    print(f"\n  [{v}]")
    
    win_c = {m: 0 for m in compare_order}
    sec_c = {m: 0 for m in compare_order}
    thi_c = {m: 0 for m in compare_order}

    for fold in data[v]['folds']:
        fold_df = data[v]['df_compare'][data[v]['df_compare']['Fold'] == fold]
        ranked = fold_df.sort_values(acc_col, ascending=False)
        rm = ranked['Method'].tolist()
        ra = ranked[acc_col].tolist()

        win_c[rm[0]] += 1
        if len(rm) > 1: sec_c[rm[1]] += 1
        if len(rm) > 2: thi_c[rm[2]] += 1

        top3 = [f"{rm[i]:15s}({ra[i]:.1f}%)" for i in range(min(3, len(rm)))]
        print(f"    Fold {fold}: 🥇{top3[0]}  🥈{top3[1]}  🥉{top3[2]}")

    print(f"\n    {'Method':<20s} {'🥇':>3s} {'🥈':>3s} {'🥉':>3s}  Top3")
    print(f"    {'-'*20} {'-'*3} {'-'*3} {'-'*3}  {'-'*4}")
    for method in compare_order:
        total = win_c[method] + sec_c[method] + thi_c[method]
        print(f"    {method:<20s} {win_c[method]:>3d} {sec_c[method]:>3d} {thi_c[method]:>3d}  {total:>4d}")


# ============================================================================
# 8. FOLD DIFFICULTY
# ============================================================================
header("FOLD DIFFICULTY", 8)

side_label()
all_folds = sorted(set().union(*[set(data[v]['folds']) for v in versions]))
for fold in all_folds:
    line = f"  {'Fold ' + str(fold):<22s}"
    for v in versions:
        sub = data[v]['df'][data[v]['df']['Fold'] == fold]
        if len(sub) > 0:
            cell = f"{sub[acc_col].mean():.2f}% ± {sub[acc_col].std():.2f}%"
        else:
            cell = "—"
        line += f"{'│ ' + cell:>{W}s}"
    print(line)

for v in versions:
    fd = data[v]['df'].groupby('Fold')[acc_col].mean()
    print(f"\n    {v}: Easiest = Fold {fd.idxmax()} ({fd.max():.2f}%), "
          f"Hardest = Fold {fd.idxmin()} ({fd.min():.2f}%)")


# ============================================================================
# 9. STABILITY
# ============================================================================
header("STABILITY (consistency across folds)", 9)

print(f"\n  --- 5-method comparison ---")
side_label()
for method in compare_order:
    line = f"  {method:<22s}"
    for v in versions:
        m = data[v]['df_compare'][data[v]['df_compare']['Method'] == method]
        rng = m[acc_col].max() - m[acc_col].min()
        cell = f"±{m[acc_col].std():.2f}%  (range {rng:.1f}pp)"
        line += f"{'│ ' + cell:>{W}s}"
    print(line)

print(f"\n  --- Most/least stable individual metaheuristic ---")
for v in versions:
    stab = data[v]['meta_df'].groupby('Method')[acc_col].std().sort_values()
    most = stab.index[0].replace('meta_', '')
    least = stab.index[-1].replace('meta_', '')
    print(f"    {v}: Most stable = {most} (±{stab.iloc[0]:.2f}%), "
          f"Least stable = {least} (±{stab.iloc[-1]:.2f}%)")


# ============================================================================
# 10. OVERALL SUMMARY
# ============================================================================
header("OVERALL SUMMARY", 10)

for v in versions:
    d = data[v]
    bl_acc = d['bl'][acc_col]
    bl_p = d['bl'][params_col]
    bl_s = d['bl'][size_col]
    
    cm = d['df_cm']
    cm_acc = cm[acc_col].mean()
    cm_p = cm[params_col].mean()
    cm_s = cm[size_col].mean()
    cm_ret = cm[ret_col].mean()
    
    # Best standard
    std_accs = {m: d['df'][d['df']['Method'] == m][acc_col].mean() for m in d['std_methods']}
    best_std = max(std_accs, key=std_accs.get)
    best_std_acc = std_accs[best_std]
    
    # Best individual meta
    meta_accs = d['meta_df'].groupby('Method')[acc_col].mean()
    best_meta = meta_accs.idxmax()
    best_meta_acc = meta_accs.max()
    
    # Wins
    win_c = {m: 0 for m in compare_order}
    sec_c = {m: 0 for m in compare_order}
    thi_c = {m: 0 for m in compare_order}
    for fold in d['folds']:
        fold_df = d['df_compare'][d['df_compare']['Fold'] == fold]
        ranked = fold_df.sort_values(acc_col, ascending=False)['Method'].tolist()
        win_c[ranked[0]] += 1
        if len(ranked) > 1: sec_c[ranked[1]] += 1
        if len(ranked) > 2: thi_c[ranked[2]] += 1

    gpu_spread = d['df'][gpu_col].max() - d['df'][gpu_col].min()
    train_spread = d['df'][train_col].max() - d['df'][train_col].min()

    print(f"""
  ┌─────────────────────────────────────────────────────────────────────────────┐
  │  {v}                                                                        
  ├─────────────────────────────────────────────────────────────────────────────┤
  │  ACCURACY                                                                   
  │    Baseline:               {bl_acc:>7.2f}%                                   
  │    Best standard:          {best_std:>15s}  {best_std_acc:>7.2f}%  ({best_std_acc - bl_acc:>+.2f}%)
  │    Best individual meta:   {best_meta.replace('meta_',''):>15s}  {best_meta_acc:>7.2f}%  ({best_meta_acc - bl_acc:>+.2f}%)
  │    Combined Meta:          {'best/fold':>15s}  {cm_acc:>7.2f}%  ({cm_acc - bl_acc:>+.2f}%)
  │                                                                             
  │  MODEL EFFICIENCY                                                           
  │    Baseline:     {bl_p:>9,.0f} params  /  {bl_s:.3f} MB                      
  │    Standard:     {d['df'][d['df']['Method'].isin(d['std_methods'])][params_col].mean():>9,.0f} params  /  {d['df'][d['df']['Method'].isin(d['std_methods'])][size_col].mean():.3f} MB  ({(1 - d['df'][d['df']['Method'].isin(d['std_methods'])][params_col].mean()/bl_p)*100:.0f}% ↓)
  │    Combined Meta:{cm_p:>9,.0f} params  /  {cm_s:.3f} MB  ({(1 - cm_p/bl_p)*100:.0f}% ↓)
  │                                                                             
  │  COMPUTE                                                                    
  │    Train time:  spread {train_spread:.1f}s  → {'negligible' if train_spread < 5 else 'minor' if train_spread < 15 else 'notable'}
  │    GPU peak:    spread {gpu_spread:.1f} MB → {'negligible' if gpu_spread < 15 else 'moderate' if gpu_spread < 30 else 'significant — FS reduces GPU usage'}
  │                                                                             
  │  WINS ({len(d['folds'])} folds)                                              
  │    Combined Meta: 🥇×{win_c.get('combined_meta',0)}  🥈×{sec_c.get('combined_meta',0)}  🥉×{thi_c.get('combined_meta',0)}
  │    {best_std:>14s}: 🥇×{win_c.get(best_std,0)}  🥈×{sec_c.get(best_std,0)}  🥉×{thi_c.get(best_std,0)}
  └─────────────────────────────────────────────────────────────────────────────┘""")


# ============================================================================
# 11. V1 vs V2 DIRECT COMPARISON
# ============================================================================
if len(versions) == 2:
    header("V1 vs V2 DIRECT COMPARISON", 11)
    
    v1, v2 = versions[0], versions[1]
    
    print(f"\n  --- Which feature set is better? (per method) ---")
    print(f"  {'Method':<22s} {'V1 Acc':>10s}  {'V2 Acc':>10s}  {'Diff':>8s}  Winner")
    print(f"  {'-'*22} {'-'*10}  {'-'*10}  {'-'*8}  {'-'*6}")
    
    for method in compare_order:
        m1 = data[v1]['df_compare'][data[v1]['df_compare']['Method'] == method]
        m2 = data[v2]['df_compare'][data[v2]['df_compare']['Method'] == method]
        a1, a2 = m1[acc_col].mean(), m2[acc_col].mean()
        diff = a2 - a1
        winner = v2 if diff > 0 else v1 if diff < 0 else "tie"
        print(f"  {method:<22s} {a1:>8.2f}%  {a2:>8.2f}%  {diff:>+7.2f}%  {winner}")
    
    print(f"\n  --- Model size comparison ---")
    print(f"  {'Method':<22s} {'V1 Params':>12s}  {'V2 Params':>12s}  {'V1 Size':>10s}  {'V2 Size':>10s}")
    print(f"  {'-'*22} {'-'*12}  {'-'*12}  {'-'*10}  {'-'*10}")
    
    for method in compare_order:
        m1 = data[v1]['df_compare'][data[v1]['df_compare']['Method'] == method]
        m2 = data[v2]['df_compare'][data[v2]['df_compare']['Method'] == method]
        print(f"  {method:<22s} {m1[params_col].mean():>12,.0f}  {m2[params_col].mean():>12,.0f}  "
              f"{m1[size_col].mean():>8.3f}MB  {m2[size_col].mean():>8.3f}MB")
    
    print(f"\n  --- GPU comparison ---")
    for v in versions:
        spread = data[v]['df'][gpu_col].max() - data[v]['df'][gpu_col].min()
        print(f"    {v}: {data[v]['df'][gpu_col].min():.1f}–{data[v]['df'][gpu_col].max():.1f} MB  "
              f"(spread: {spread:.1f} MB)")
    
    v1_spread = data[v1]['df'][gpu_col].max() - data[v1]['df'][gpu_col].min()
    v2_spread = data[v2]['df'][gpu_col].max() - data[v2]['df'][gpu_col].min()
    if v2_spread > v1_spread * 2:
        print(f"    → V2 has {v2_spread/v1_spread:.1f}× larger GPU spread — "
              f"feature selection has more impact on V2's compute footprint")

print(f"\n\n{'='*130}")
print("DONE")
print(f"{'='*130}")



1. MAIN 5-METHOD COMPARISON (mean ± std across folds)

  --- Test Accuracy (%) ---
                                                                           │ V1                                                   │ V2
  ----------------------──────────────────────────────────────────────────────────────────────────────────────────────────────────────
  baseline                                                     │  85.71 ±  8.97                                       │  93.39 ±  4.68
  mutual_info                                                  │  74.89 ±  9.34                                       │  93.97 ±  2.97
  rfe                                                          │  91.41 ±  5.90                                       │  94.55 ±  4.73
  lasso                                                        │  81.76 ±  5.38                                       │  87.01 ±  7.23
  combined_meta                                                │  92.22 ±  4.22                          